# Crawl corpus ViBioMIR trên Kaggle — tự upload Hugging Face

Notebook tự chứa code crawl, không cần clone repo hoặc upload file Python.
Phần chẩn đoán đã tích hợp: Output tự tạo `<run>_diagnostics.zip` để debug.
Bật **Internet** trong cài đặt notebook; dùng CPU, không cần GPU.
Chỉnh ô **1. Cấu hình**, rồi chạy các ô theo thứ tự.

ID bắt đầu và kết thúc đều được tính vào khoảng chọn. Đây là **ID gốc**,
không phải số thứ tự dòng. Giữ nguyên ID/URL nguồn, không dịch hoặc sinh lại text.
Trích HTML/PDF có text; Selenium thử lại URL lỗi, trang JS hoặc text rỗng/QC lỗi và render JavaScript. Chưa có OCR.
Dùng chung `priority_probe.py` với notebook 97 hostname: header trình duyệt,
timeout/retry có ngân sách, giữ lỗi theo từng lần thử và kiểm tra challenge.
Tài liệu nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).

**Bản tự upload:** `crawl_corpus_kaggle_hf_upload.ipynb`.
Mục 1–9 chạy quy trình crawl/xuất; mục 10 tự upload output đã hoàn tất
lên [AncientWings_R2AI_STAGE3_ds](https://huggingface.co/datasets/AncientWings/AncientWings_R2AI_STAGE3_ds).
Trước Run All: mở **Add-ons > Secrets**, tạo secret **HF_TOKEN** với token Hugging Face
có quyền ghi vào dataset này và bật secret cho notebook. Token không nằm trong code/output.


## 1. Cấu hình — chỉnh ID tại đây

Để cứu run đã crawl xong nhưng audit/export lỗi, đặt POSTPROCESS_ONLY = True,
chỉnh đúng khoảng ID và RESUME_RUN_PATH tới thư mục run đầy đủ trong Kaggle Input
(hoặc giữ None nếu run vẫn ở session hiện tại). Chọn SOURCE_PARQUET nếu nguồn nằm trong Input.
Chạy các ô từ trên xuống: mục 6 tự bỏ qua crawl; mục 7 chạy lại audit/export.
Snapshot crawler/runner/probe, cấu hình crawl và manifest nguồn được giữ nguyên.
ZIP diagnostics chỉ có một số mẫu, không đủ để hậu xử lý toàn run.

Mặc định chọn ID **583–1582**. Để thử trước, đặt `MAX_NEW = 10`;
sau đó đổi thành `None` và chạy lại ô crawl để làm tiếp.
Giữ nguyên cấu hình tải trang/browser và phiên bản code khi resume cùng run.
Với khoảng lớn, chia thành nhiều run nhỏ: crawler hiện tại nạp danh sách công việc vào RAM.

`SOURCE_PARQUET` để trống sẽ tải nguồn đã chốt từ Hugging Face.
Nếu đã thêm Parquet vào Kaggle Input, điền đường dẫn `/kaggle/input/.../links_corpus_vi_en_zh.parquet`.
Không dán link `/blob/...?...page=...`: đó là trang xem dữ liệu; ID_FROM và ID_TO chọn ID thực trong file.
Khi đổi từ nguồn cũ sang nguồn này, chọn RUN_TAG mới và không resume run dùng nguồn cũ.
`RESUME_RUN_PATH` tùy chọn trỏ tới **thư mục run đã giải nén** từ notebook này trong Kaggle Input.
Để nâng cấp run cũ hoặc thử lại lỗi, đặt `RECOVERY_RUN_PATH` và chọn `RUN_TAG` mới.
Recovery dùng lại raw 2xx đã kiểm tra hash, chạy lại extractor/QC, rồi crawl URL còn lỗi.
`RESPECT_ROBOTS = False` tải trực tiếp giống `domain_samples`; đặt `True` để
kiểm tra robots trước các request trang. Trạng thái robots không được chuyển sang Selenium.


In [ ]:
# ===== CHỈNH CẤU HÌNH TẠI ĐÂY =====
ID_FROM = 583
ID_TO = 1582              # Bao gồm ID này
WORKERS = 4
DELAY = 1.0
LOG_EVERY = 300          # Tiến độ mỗi 300 URL; URL lỗi luôn in ngay
ATTEMPTS = 3
MAX_NEW = None            # None: toàn bộ còn lại; 10: thêm tối đa 10 URL
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
RESPECT_ROBOTS = False    # False: giống domain_samples; True: kiểm tra robots
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
BROWSER_TEST_MODE = 'failed_only'  # 'off', 'failed_only', 'all'
BROWSER_USER_AGENT = None  # Dùng User-Agent thực của Chrome
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_WAIT_SECONDS = 15.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None      # Selenium Manager tự tìm/cài Chrome; hoặc đường dẫn có sẵn

POSTPROCESS_ONLY = False  # True: chỉ audit/export run cũ, bỏ qua toàn bộ crawl
SOURCE_PARQUET = None     # None: tự tải; hoặc '/kaggle/input/.../links_corpus_vi_en_zh.parquet'
RESUME_RUN_PATH = None    # None: mới/tiếp tục session; hoặc thư mục run trong Kaggle Input
RECOVERY_RUN_PATH = None  # Run cũ để reuse raw thành công, thử lại URL lỗi trong run mới
RUN_TAG = None            # None: ids_<ID_FROM>_<ID_TO>_synced; đổi khi muốn run mới

from pathlib import Path
import json
import shutil
import sys

if sys.version_info < (3, 11):
    raise RuntimeError('Code crawl yêu cầu Python 3.11 trở lên.')
WORKDIR = Path('/kaggle/working')
DATASET_REVISION = '71be0e1239a19dfd7a717e2bc13e3bec21bd0334'
SOURCE_SHA256 = '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7'
SOURCE_URL = f'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/resolve/{DATASET_REVISION}/links_corpus_vi_en_zh.parquet'

if type(ID_FROM) is not int or type(ID_TO) is not int or ID_FROM < 0 or ID_TO < ID_FROM:
    raise ValueError('ID_FROM và ID_TO phải là số nguyên; ID_TO >= ID_FROM >= 0.')
if type(WORKERS) is not int or WORKERS < 1 or type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('WORKERS và ATTEMPTS phải là số nguyên dương.')
if not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if type(LOG_EVERY) is not int or LOG_EVERY < 1:
    raise ValueError('LOG_EVERY phải là số nguyên dương.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if type(POSTPROCESS_ONLY) is not bool:
    raise ValueError('POSTPROCESS_ONLY phải là bool.')
if POSTPROCESS_ONLY and RECOVERY_RUN_PATH:
    raise ValueError('Hậu xử lý dùng RESUME_RUN_PATH hoặc run hiện tại, không dùng RECOVERY_RUN_PATH.')
if RECOVERY_RUN_PATH and RESUME_RUN_PATH:
    raise ValueError('Chỉ chọn RECOVERY_RUN_PATH hoặc RESUME_RUN_PATH.')
tag = RUN_TAG or f'ids_{ID_FROM}_{ID_TO}_synced' + ('_recovery' if RECOVERY_RUN_PATH else '')
if not isinstance(tag, str) or tag in ('.', '..') or any(separator in tag for separator in ('/', chr(92))):
    raise ValueError('RUN_TAG chỉ là tên thư mục, không phải đường dẫn.')
RUN_DIR = WORKDIR / 'runs' / tag
SOURCE_PATH = WORKDIR / 'data' / 'links_corpus_vi_en_zh.parquet'
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_PATH:
    previous = Path(RESUME_RUN_PATH)
    if not (previous / 'manifest.json').is_file():
        raise ValueError('RESUME_RUN_PATH phải chứa manifest.json, records/ và raw/.')
    if not RUN_DIR.exists():
        shutil.copytree(previous, RUN_DIR)
    else:
        print('Đã có output trong session; dùng run hiện tại, không copy đè.')
if (RUN_DIR / 'manifest.json').exists():
    if not POSTPROCESS_ONLY and (RUN_DIR / 'crawl_config.json').exists() and not (RUN_DIR / 'corpus_config.json').exists():
        raise ValueError('Run cũ cần nâng cấp: chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    saved = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    if saved.get('id_start_inclusive') != ID_FROM or saved.get('id_stop_exclusive') != ID_TO + 1:
        raise ValueError('Khoảng ID không khớp run đã có. Chọn RUN_TAG mới.')
if POSTPROCESS_ONLY and not all((RUN_DIR / name).exists() for name in (
        'manifest.json', 'sample.jsonl', 'records', 'crawler_snapshot.py')):
    raise ValueError('Hậu xử lý cần thư mục run đầy đủ; ZIP diagnostics không chứa toàn bộ checkpoint/raw.')
print(f'ID: {ID_FROM} đến {ID_TO}, bao gồm cả hai đầu')
print(f'Output: {RUN_DIR}')


## 2. Cài dependencies

Lượt mới dùng phiên bản tương thích mà pip chọn trong môi trường Kaggle,
Phiên bản thực tế được lưu cùng run.
Khi khôi phục run, notebook cài lại dependencies trực tiếp đã ghi nhận.
Nếu bật Selenium, bước crawl tự kiểm tra thư viện Chrome trên Linux, cài phần thiếu
bằng apt và mở trang thử trước khi tải URL. Kaggle cần Internet để cài package/driver.


In [ ]:
import subprocess
import importlib.metadata

package_names = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld']
if BROWSER_TEST_MODE != 'off' and not POSTPROCESS_ONLY:
    package_names.append('selenium')
requirements_path = RUN_DIR / 'notebook_requirements.txt'
if requirements_path.exists():
    install_args = ['-r', str(requirements_path)]
else:
    install_args = package_names
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs').mkdir(exist_ok=True)
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
)
with (RUN_DIR / 'logs' / 'pip_install.log').open('a', encoding='utf-8') as log:
    log.write(installation.stdout or '')
if installation.stdout:
    print(installation.stdout)
if installation.returncode:
    raise subprocess.CalledProcessError(installation.returncode, installation.args, output=installation.stdout)
versions = {name: importlib.metadata.version(name) for name in package_names}
print(json.dumps(versions, indent=2))


## 3. Chuẩn bị code crawl

Ô dưới giải nén snapshot nhúng trong notebook vào output; không tải code từ mạng.
Snapshot crawler/runner/probe đang có được giữ nguyên để resume đúng phiên bản.
Auditor, exporter và diagnostics được cập nhật từ notebook; không thay cấu hình/hash crawl.


In [ ]:
import base64
import hashlib
import zlib

EMBEDDED_FILES = {'audit_snapshot.py': 'c-pO6TW{RP6@J&Rn8+<irn}T7j?qe58IJ8BM&iV9oS=ZRU`FKZ?nJzKNJ`pB0$SvuPX!vJKoJxH;siyI21Qe}Pu4>p6y$%Xzog%p8FINd$th~U(k^*!bMEJiR##Ww84O~X3MSK143#X$Yhi|uX~`nVhiO)nGAtNRBm8Go8uNV-$0A}%97=J-ijfc{8)S-c7Ak%eGgYPj>gwu>O!7>XjH_YJRUuXeDoa?H0XZyXmKMwc+|4TRQ#JhXW0}{VN4yxta=-anCNblM{vC)S<hiI{eN<$r;gIu^aBAw0@U;d~I7OB;-x}m%T+|<{REAk3B3|+pLoYrC)0O}=WIpCfWk)^^_q2ivAXol$Wo0E21CZvCOM-Y?D-8dXC@Ym}A^kYxQK4V@1QwLyxO7Ddt}-3=oT?mjf9ZIhr4(jK&co6rsA*yjbNr7onYt~J{WMbvkLAb0b>4pcuG3)}&oA;=mTp6|7;%@trwLkR`*^$g>MaA{kHm2#hoS&>eKvj+Xakf=2zNh=#vN9RZwW?13&eViUi;>2bH)yHl*nTq3dhsna=_B8tg-x}R5Euhsg`_iXQ=?LNb+(VkQ7lCRtX%{(U7GY&!Lc<K3S1yrbE}V@H(29h8D-6$V=vGP(f$%ACbGN`>?L+u=jaf3H^DgL{+LCobShCMlPeVgu_Y}TSF>nrVb1Ljv8uc!xbVHA<j<`(hG=G5O^(2pGQ%^EsE=OyIEDj%t}11q!Ll@iK@hv@Fr}#1$>0uW1Nc~BEJLe4|o-qy^ZztEAfl0QX!#^M4b1W_l1%J30s@z2saZAO2k>nOT5rQgw&D65e{C6qQKFt<w3ZBE-Wkki<lLzd5yE-Ek&?PPknqhFA|>00h|(Vb*K^G8guH$J_XvuxyapXkb)<9EHpsucqRh($E5&EkG<4uDDPRwz(j-7ZyBG<LbUwCb>7vWI)GRri-ecq$g)3@<%lg|p5-E)k9<lba96>T2}^^B)9$rW2xV<gXI`7XEgBt}p45TVuk{dv-HF<BWaRWUp3+E~_L)9CL74B}Rp-4|A)k<SYr!7eE9`m2agOw^W4~s##qgn0@dBnA#duCY8F9+m)}w&ZGK%rOeq-Cl<U;}Xw4|)fT4yjh4CH5$oYz2TL$yRP)^XJSVvEo}I?X=Z{D4j*QNrAqC;JiKve4SQ+>Q0ko9tyqzmRwKowiuh;`vpMq>G$NG$)ebl=giq8PE9Y)Vcf93>Nt7bgVA&TdRD;=2KgyD80EY1$Ed!#yE7*+jGgzNQ5%Q&sajf+6_Ff@A+jG<YWEVYrCSM)%ab5LKeT$cQoD^;ng@V5S&pa3SDOODQD7^J+wl3P!VOMJyXF=c;$wx0;496SWraqFwXW}=VhC7Y_y?dS3JHZ2rP}WS}7K(buLl`>Bd6N>*}h5Kd#`FDK!wWD0LiBV~1K)pv!4EAau9c5tp`yi(?j!M0k*+vncDJ4mr{`X{Fx|SgIrR^u!!s_E;0q2%X?a5n2zV5UQI{$3!o9@D#^K0~!=1FROyynXsODjxcftF&`4dUjM8eC=T^Ij+R2Vtd#n}KtJvKI@b!D>Fwhe4VuB~6F^VL%2s3Ev(>41wQ|?;vR)5;%W<hVsoc?$kt<p->KQ1aW;4SA86HZM&vZyl7(*eCa3XP?6wAm{Zw9kazpY!eBv&ZTwTFXbgV>)blv8AEfhB6SUI}rrT9+K_F^=;Bnr$_Uy0L_QRasExcK4m1Kf3pNcTsYMj;K#C%D_Qmjz%e{>#n2ojV-=&xYQ4JOn(xP%Mqc){jWEN3rQ!4^J3MJv1fg`@ADibhvtSF$y<oj4!=FCr9K36TvDUDJpWTVTss(D{0n}fX|W$Z{qg(D^RGj8@12L+>{ukN7-xqUUut(6UY>s%`rf<_L|ll);^XkrBnF|wUpFDTh1S*3iR2;FOJQk8%C3lHnW=7$sZvL&D~wrSl%lYvaGH6gzmO}0R@qPEiXLC(|2xMp7HD95%`j$tNoSP~p?k{(ZJ{+2oA7n!vEsSj<QV2`OXwMIvF6v1K{$ee9kM=^1^w(QT!Tb$F4UojG@_9byv-@0H}izdJ{gl1IJW_#TxLE|Sk`0~`^!?7w}OE}FsR~q+!E1S#@;yJaz;oEGSp48f!tf~Un7*&tnPGR9Bd^T&01pqG37(o@z>Tg)H2J@TPxwi>uJhlcoju7^#|JTx$(b|5(CGxyBBD_CTPNdn;@8iy-owXEO|JhQ7=GEHjcGdX$}=}FcMH{!E_<G3P-MTKJeW<8k}NOJ-&T<92dv8J@@w3&rbJl-{^aYkBY`q|G^c^Z9}L%$74He<JEQTSIbqn-0WBQm?oej&yLGQkcBER6S1>g?>E?kG!jIFwqmB^hdPC#>TBj(N}x!t9hQniOu0jm3N@L2%aO3rMcn<RjC2}^+oqiOhS>_7(x-A93;G}jiYNLLwsiXSqVhML%Kb=XxhY-C54^O(e304<R@TF%mbskuKE2b@?$-b==C-`HTwUqjH03S4_$M|Bv`~Z;uU$TWhR*&Wm)q<>UY>sf#VbCkZDfXw)lE&i(vi567GaXsr<$pj$3K)IvUD~mAJF1573KZoJkFF*r%91@jeY1-X+G|}^uZpSM-EPR^n=~g9qbf$?RL&a$Fvp}dvG=wNa!O|LBiERl?U*t17}gsH`_5&W9IOXR$a)cSMh>G#A}9*dD}s|-7@49BJ^iRqymUjuVzq+`aJZ-4j2^Gl-ler*d1uU3Uxiv!v}pG=&uv9W*DgwC}XVW3*!oO!1gAiLydqLg6cQ*JcU=*#}dedZZJ5s`z!#`Dy=i?o4U6}ZZdg<dRAnSStRC6R_8%IsYDVjfH|l)Pqy9}76G-hrn&<Or*lJNirLzY@G=7})#w>@y&!`Ywd+~Y$$m1Mc1bO!nJlQb9k%YVUABHbn<B(|2K1aeRhcC^)50)oNWtw}i*2nR8g=Pj0SicGhcIj)nRRX7nWsLguv)QoA!z%~x*0a>B>=C`&IZ;4?E=6*!X1h!Gp=6NMOBg)t_7k|p{NwLoJA;Q?{$v+*TF%lvh+5+$tJy72QFY}f~ejiw)i<VJGc0%MI1f$3!Td+GrQIE683fCn8diN=H~sYvQMv_eZb~GjC*ER##FNrsCO`0>$q7d^sq%YHvVJ%%e}dYMqLNBTT|;`WK7Mz&|=bNyJH$G3**@YWUi`utXbh`cr3iFF58z88WzbATdSgd7Xe)-x(B;{n^DR#7Lz50`AR9LnX9+vNA%8uVu<vmpyU!DLn3Ie9csr+F|`Nl0k&+Lt(Vkp&=z>^-694F=CY8-L0TpI*yv5=1xwIqOQ^{cs7c4P<D9jfp)ud2-!sbmzc{Ma|58UK1(9)_otKb^=yF3}0qySSo5Pp`e)UkU?(QU_#C0VNPonItY7%&=O|*N5f(#`|<yEn>W}eL?3C9+4n*L7)7~{4!Zso^xj|lO)mE1<G4`sTwc{4xWCM$GDLJmg&^KYF+;4S0gWFJu@)Yb+eX0eRejW7(ib6mLKTw5EPK(%D(Gjb0uNTfQ6BxY4U-ojmNv`s&{;E5-t=;Cs<O44F$W1txRZx4B{ML2^!PmV^=UAMp_Dz@;wd!%^2ouPCM;_OI&+v4f?XoR+Q+Z^d;`$%U#6|xhs$kf)2U)_4;Y{zt>^ST*r>^>UF%jf^D+I<+}jFlH(N(>Q~=fA6(q_VRH_?>+B;pOx1DwYn-uaUg?E0(STN$2LK&+cBHe_n#j#WUbzVWB&=^78yU>blX`U3`_YeIkxX>ab8l=cCK#UmI{^OrQPmhl{Vqu=|hS^ZNAx<SbDP@C7d}pZ_Hx^lvyMp{2<e65ErDXX%Iy<;639XD#1#`m_7a#^on@0=IZ{+c3GP_U(Oj4M471H$sEe9qSakJGM~lHv2F3pWpnEodDaNN+f3QKt`|#P@#dc8HnEHyF1v1Fum*OIpjVLIGFo)1jQ@6q`h=lU7ml2&B#QJ9jhbs&7URAu1`Ez>J_k2!Z2!g=UTE!F8sAfCVjUNlh7b+&ATncU2Pp~W(*(1{6vyjrt_<pdD?3@GySwGU7h5HrjrI$$_q2)wN(-5^-OVXh99D|GzBG*PCz%<n-ka1?4MW|H`H?D_GGn-<@?%ukKbYkqs#L@=d~fQ`)v29XKN>|X7<cqZHJrFWwi7=tV5#N^tA%--8b%Zv5K8;bcp+XtM4IT{|;2RJuc=dvfAf4)A_=r3wC0iZ11`@vJYvmge8a8fl8yBHTd+ANV3;zh<$`5<~$(c=NnHRzSUCFGCtEiXa8)Te4U+iJJqPWiDK_;VWQf$K|yjg$-$%yuppbZ4RiP6S%@5S@onC=MSAh&m>tur6o`LIS+0<RQ%;KnG9nP3;gW$Er!3KFtUwI07+YN$Ywg$zfB6=_(fW+YZqL6NTl1wrku0k45~}+Le(AP1NBqNS3*VWO^OSPy7wnY1OMKerJ|*B&*6ntg{<(!Gee2`(m^c8y3A(v6fSl+i9-mCP#?y`&>{{1nX8!SY6?f~tY(j320z+qy59eCl5A_j0(Q15fZ<~EM6Rq`o`uY2?$!dp9`bS=WPKi7n@oF1jIOT8Cc6KM&Y>KX7Y_!&&&J}mwyZ7Kp@c90tcOF0SlW1A4NTTJ<3jQSJ1!e^O7ajD+H|R`*AjQTf2y{ao1oVd`L14#uRBG&6B8cZx>PQE22qNy1ET#V>W#zxxsz);',
 'corpus_runner_snapshot.py': 'c-o~K+iu*(^?ko$syvYEtSC9M-Bb=A6kAD*IF>8R35xB4p~%_YiMS-gAtkG_{-J#+`r05U0<;KP^wECkgM<8s{-o#3g`DAKB_#^r-JQ#sGv|JpY;0^CRw*f$l<;L((3r&8Dr70ii|m*Znl4yMISJF4<kfM)_>w$%yqOo2)8dq+3le2TUU3pFX>^ijEG@aWv9U2?t2`@85*CX*EI1v_i);mJlY~Yk%Ti9%f=5}Emb93Vn9jp0DPtCudUM22^;h|kuw$<(SyGQ?`r9`=OZE3vST5yxIeddX`mE<Tp}zBTJ{o<wfB53b!876$M_yyIsM3@co2NU@=<)t1doQ0Jy@1(^5rIFaf;$J{f~KXjOPnvVKd~eUx9)q}#Qln;adyVZvm<hM+uMFfVBo>Mhve0Rdt<Vf=L!9a9)Hftt@}H_^>!W*_w&z=zIZwz2|J<WQy^V7CXbeYGu^to0|4+p@*<pv1yfs`i9{l%Nq7$D-}NTM3CohM@{+^v9r#(Y70oKdZu-GC%-_53&0zSriWd||?|ulQ5)3df2UY~Y<xK4VBuqHflVUK)a=|K?ywF3f1J!7h0o4FlbN=1?>Y$!Ky{n*MY;4&<jGMW#I1g&Xe`n6kLzYwp4azc++b$K_#|5weShUJwiX6g)G!C*fId>+_w8h3}SxR9p60{GFSsE7Sdf0}VWEQ$WTLb1<$jX3IAVkd7^_xk9Xdg^Qm!r`Lgq?`%^Kb>S!QH3`&k|Gv06yLo2d~1E%_%QI{?x1|P(V?4VHw#Xj$Y&8pfeWx*t}jphx-T1kT3Omd7N0CIAih~zdmKE<^r$+AKZW7%yw(`6(Qr4d>JMc-7ku)aGe)oDM84f_+h%Bv8PTyW95?cALnH`O<kHsSqz55cdBx}`5|Bta>7fX#;Q3|kp-?k#m(a4+)cm~jmbPK2!11Aq~t!lGLCrgljZJsrs<Htc^AdhVX-q4`wFp10>2dZ$0cfDyza`6t2|**SW>ketRUBS24Ss=8m`IS0wkh{E$&VLktD_+a=khgO%+^M5>BqbFB=zfmAu2HAmxU>UYeC+pGRLYUUJvGfJ6z!SfPlB^okNfs3pgh3q4Rf7mplIh&fUfz(+wh*ThTTWhxfT))8{<JgPkf&~WSJ;Zw3=9Bg*91UrMsT-s!n=PRlzAJieN@K%bx7F`SaZ6%dxm?4f@#MLx+Y9N{}8Vuutro)6*I=}urZ5aBG3!{i~K2g4R0veWangT>|Nl#7(04V~K<tof3`sA_AZ}g|Ggka8iRUU&dy9yXD!?NPOeX=jNNzTh5N6(G>tb0vH1y22z0OXhyD79=^mU-`5W5QW~5=Eg3PQd%x5Snr8L^0g6@|sg5%>ooof4<R{maRS=ttCgX#grJ60a_Fk8Pv}<$@D6dV%E}YDQ3u~qJp(wQ-Ro`sju7;*xnltoysZLSXQmTZzsud7@eTwhYh|IXAQbFtL1uJ6H*Eu@gZ2#Ae|wqDiS&LRat~lNn;RP@XaOo{GdF~sqd69wY4l)$wTyD5TTa7^hZP2&|J5rM~?#h=?o=Jgea{wGQ=8W3m{tniV!ArG)9dN?}cD6=debF9)1DgbnMy!ZZ82$x5wmtVza+z&4lbI;%lFAR<M#<@rO2GgRV-uk70{0fhs#xPz=u3nRV@@^DM8DP^J=V0wkVQMMMLDgRzhblco+Bwi2RNNIkeCqXiIvDqjbwR#aRlRY-voe)HN@wUY*(flIu&loWyGU{i6Oh<0XHA_Nf+Xn<IV(Tj>anh>Tf{?~cAK5t{h5^}+=C2FWOEUjpBQg((J(Fr}@HA4gVEkiT-f*86M6ch=-0HXu=kQpX^v)m63)NaOU#!TC0>w)ua6(+1aHvs67c@i!pL}A(V7wHmAm){S#&qrC?2B4|aY+$yHH&c-objv_H<I4_%va@L$+Dzo;V#cQC48t_%%e{l(@ymlpj#wtd(q=I19PmYb!t6|1UPR1I6_saWQ%t4~Kn-C*0DPC)21xm)>I%`(W<!a^42_QohkIcTnFE-$?97tc>o09zRjS#vecr4CUmVKO5`i@PNw_+WLsIM#ceCid6j(VEeU))2>zNlU0~^O=I&-1d?W#fx>O%+OZih&E@piu}(77&v-~memsAB*SOl}t767qpfk|2l*8XHxLaf@yUMKE`oIyI5&TuMSWB(?d&H5zNN4A;vOJuvwzt`oI}K);ygiqyPXtss9#`y^rn;rVy191nB-48Jy0$ZZ@)8GyvJwSjCJi`v&z(#MVyEI7dNZqb!U-%`VxbOR90Q0tTy0!sy08ONC&14;e=wZ}^NQD-%BBlfkqcZ%s8un&o*<W4ezE7~<2Rgy6rL1?Tc=cKQkXhvQaepD3{WM~@9*(-&C24G+y-6x|-EEI6a6?H}I*ZZeic&OTy)c&gDp@zIt`^X#ms9)@tW_3W4hAW89ka8#;(<U4aP62HtpdOTQoymaQ=|9&wS!U@XI1RHPuChpNnijU;3afvO&7X~p+LbC0`A{nMUO;syT|7B(Q+kHwBHwWucOrxf({?!6dkp6AD>2|o023)dcwg>IS!!>U;<<$I3wHHS;tgD+Rx_<-J1nymi-P0x5)?~Hw<NASA1$)bIv3(dETB<Wmd`dhTR6)4C8AP~BK{tE^zOTeoY7+r$CBvXe^ZiNz5N>_Wl^1z=<3anIr6nFJI>Glu1Lw={ZljqGE!c>{VSCCu|^YJz5O917&jwc00+25l+J*CXIF2&Uy=Cg%@3*92YJ5n)1Tjc4^Xe({zxurgL=7jVcObDJnA7iefRf_{PdTf|8w>B`-og9V=y(1?d<ZQx=}N&Oc<md?OfD3s;}L^g;OB)yL$7Fk|bgXE`%rK;2B5)lgp5ildCuXE6KMNT#pn;m<81I>5BaR<=)dLM}G(o_n#je9(k*{_tws9@>q6LUX!DSmWpgJh`%&rkf7kUVf3jOIaIntUiX0A+}tGM-(Bki);8LSPL_yHj<$u3UI<yyanF3?HMzjmjYGX8%~03(8nZodhnyC5+qhcg+?6X;urb3D636;|$Royl(RTpOzuqvV_<a;(B<Z6i(1r;ve0Fs7e4@y;x_a{+&<lC^LfILtlcB`^OHz54eBCuEUrRR4M%U<`^VR;x!T#al!Ql(wE$;ba>UiHk(NYwlc}okm2I@kvlt^6&uG0R57E-G@TTIBE2|3G(6IyWp-UJHruL3YozHCgWC;<G2NMzM$k}U#?!~V|pwn~T*abL8!@D9pziW4v#x^qXjv*5fta{RJx7l@xzyMHnh83$%^PBx!AlTJHj&I(APX&5b)?+qXj(ajm*HlU*8tSzd)Oi$A6EG6c_Uem_FN{y?LYc;H9W30s@ipN=&tUJWsNLCe;aiOz`+w@sx-rC>q8EW+y*Hbwfb{{>)=PU)W*VqH_nFykd-{d2oY`1c;ZnYc}?aLZz2~~DuGqCOKbdX-1=2Js4EWJ0l^sFQwk-O_i+TKy)7uIOuA#?%SrO<0SI>UW0d5;*1bPykjARc6#Lka|8rE1k;;6+QQNGvecR2nqub{It)SNF1Zdm3UvAif7ZA&H|_uiLIi)`1baz7B{ifYu-`n+BR*3-<syj85F9E!gOFVD*et2Ro2+G7dIn)7aC~FKRUoo9dUbP4s;<dv-hb?TX1G^)vvqdP|ui)h&ETpa~1o0)~zpzI+xO?R_eQ`f&fz0dN}pDhxi~JNnEsLNuBfg*Jhyby=grCOU+?blYXhvA0OFW7oN(c8sr4Kj$f3gwZ*g@ZFnYOv;Q@`JxCLQ#O#q7^I!lbnWhVxrb4y(p95(J?pGIJoxzFDA@aSA6_B!Z}B*EF(J3#0)uvUJl&pI<rLf3*tC(d$|)9!D-nmyt0d6zeyh}|w7cWJbTv4LS>boPZUd~Yfwa2a17iFhL{8v=ky&*PEdndlECh+?GE4;^LgEBktwdsnoGYjZ*H)qw5_>iYLe59#DS61iiiT;HLT5O^tLFedoMR#prosXiMI|0lfN?C(D3uRwR41c(GSfV-iA2e#Iro~fuR!bOEzx?P-AWZ~P@UiL?SbnGf4vc&Zgee6e~f{d8-^EMb7FUg;K9*6VT)zilC(y+{q_l_+z;lRoj>a#NDb3|9&!tCW)RWA(26jO+G$hJ=XKXt+PhlcItUCXpbjk6Cw3R$`B06Cd`tu(x%TAsOcM-rFWW<Oi`b-n;rfnHJr+-T8dAs?K0Sx%I9Pq6n3-{$KNJ`09-Mx?)GV|gHT4|1);IxWQ}x4dsSXC@*to1xna!91di~92Z(Vw+!y;{9v1%E#rd`XR0ydKYeHss`U7Jc>eS=^<JEPb_>b{uK!l>I2R(l!~s<9`Ul^jj1Zg>XHc%7Pvs-@lMZZZy*LbZd5v{Dr7QXbABVT@zA2X}hP&0Ts<xip>#ussX+;OdSrt8PAP)2_9vC*qOphs-fxCgGql)}dV&WmRqtninH$P`-)5QQT>_B3nIL`;|iNLuWk(7(A0}r12NMNI2d8J_pYhS4mx9Z{2jD3h2njf2SRBqh|t=2UB&s+4#l=8Z+m@emWR9&>rvCuiJpFS=qjR&;2iZiy}~UOe((gp?XEf0}fL91qIse#m{yPEy4PgF7#&Z@-}X;{~R;v?f|ed@;+zQ&2alq8E}q`;cwbfHwugfBz>)SfD|7@ycIy*APoGap`e*SsOJ~@(YE&t>)m-6QM4&O=Aibzrr&n!R~(~3GoygsLL++1ZbGU+Yrzh=vXm}zSV=T=hS(IaUX^<|8sNimnURB27ty%6UB1i}P`mr(xP*YPT>M9sKhC+D=2HZReQ^1^#z!RGzy$gVuuzE0539JScF9X(vf|b+NDJvkYWD3F3&?}g^0}CCW6C3VAB$J=1E|?WP;)SLdf;AzP;GIsOA7gxcsRAp81U7f%A{LI8NItUH)XX<=2$c~I5n_dYTK>0s9+ZMh__rIKlQ5_|K8f25KL}DFlp>uyBySWQ>bU}QCNFjw_E3}n=-vr;Sj74;hx+hK5!Rz_0xl_uH)8~AzTJP&8ju5F(tgI{uER++p?xgg2q0D_g%}Z^zw5@ze>RWy`vw9m@}ZL2EuuuDq0<2eI;(ZlPBxkouZPsome2bkDX+lsfb~nu8i8*fS5H>0=qgcW;->9#@vc|!{&}g5Hke<Dr69dhItT(N_L<UQdC=B#&8V*7I6iW)P2LQ9_*tsj{Xk{A&dw',
 'crawler_snapshot.py': 'c-qZ<ZIc{Fnc(;S6=gYm)h+k*^o%5*JyIKFS%AW_?3HEiHbzZR(_KAXp6;%8RrS0unhz|&E^zn)4wits-B=qRu^X^pc9-K8i#tBZ;s2osWSdX-7w&n>tjwyco*rTD)&ZmHs>~<fo_z9o&vWO_eKCF^PF}uo+Z&|e!DKC&O^&>IJW2A7cPsJUTtw-SHwky5N!IqJ$&mid(=eXJvt2Jqy?8c^4&k|T?%cV?C{3nb5R4Z2B8`H;i>LD>&Ao6oOY$&}lUdeisCVh^JWR8Q>kN{~BpOf&Rq0x?nB`I0_J+|YTuky|Jjhky*<g{T(Jb$9OS;aTaT<lgSCeFN{V*CV@&u}fSujYZ^GTFPLtQ_Q4)aO8qw2|<X*dgaqm)Yx!#v94X{5@i-!Odm`;}xSU5}<=Jn1a*c%r*O1H<w`o&>6%8u;#4;<<VrhuIjqp?>l|@S-z~@(?P9`pY~V4q>2~dhte<%+&KV%*Wi9IXoG|dKEv&a~jTu$yEQ+-O0w-b^R+D>_xf$g^QNu@jy459qGrM<{DOBzc2>5n8gD?78(NeY>`fITr@MPd+_d!B%ZbLA)8O)+$x$TJ4sIEQ&r(MzWQ7|iC(4G8kgorVVWifu+%UErI#IC{6!wo$3giczy}o1>EoNl<iNWuFGiY%qj(YmrpaLP$xwA^H_C(2WY}nY@%ru8ZoG2K>v{hEGk)XstN5fg*X<~a!gMgchI=eZ*Snn!zj6DO7hbs&T>bp@TX&X}^&5=_>;P|(51Os5h6n$p5p-py5wbH&4w?!l0TBnSPMjs9Bn7-^wxs$n-i0AGaY1Y-f`XlF9By8Eh6{AY(V<jO^<e^{0LwNJ#{|SdoR0~~_+=-VN3*7%?)WX7&=D7Mv_-(Afl=G@xozLoZkQ+2co6Iy<xxhhv^`@C_yXpx=z@NWD~kkTx;Ko|CjZEKchUvyU!Yf69wdA8ldI&>bPj;1CXOnYh0};YcJRr2+8g<uV~z41T=sYfFq^HDPClLc1(?o38pF=#9#db7PiZutgo8->$|L0BxSwTw$<7-xJjdX%TxYnL&a)<zgR#sqB(-5Sh~wVp!bt|>Ly|npdz-CJG{ark^cVSP?ODI2nw%uzuxQc72WkKdWKCmbaohot$R>e=Dd?4fV#SH{61G-WKL+3L;J$AfV@X{jX~DGncW1r_xa{Fc!*J_HHBJC;fOy#BX)5AEpE_8GRun+`fN_)LAWAj1&2ZC9;;%$a9IF7rN5Bn(q;-mN{;khl^V?iTK&nOd@21fCHVwggtKON=)2NdzcABZbeK+^+F1p>Y+kYAhdIG&=4oa84J3GG6K4E~mF`nJSPL`+fyxF4ZJH+WjI31o0>OiZd%E<WV&`a1VO*ov@`X!58?J#`Xf#2Mfc;q$l<GSa=xfzdO*LHA4VXU;lGDWVa-(d`N|BK;d5nTtun>PJd7CDe*FPtE19U*D(LQes@3?p3NK|zx`7$*t*2)$W!;H|AeqaB&NELo(3$m$=z5<)P-J^g0qRi^TIax~5FFogN2lMTjjh6j8gAZ}#d6FqO+kB2y)fHi&}djTvA(1%WbG><YMut5wg127BCGK5haO@>XZ-J-%vXW{zc9N3-+juVFF#Y5nhh8`{p@K?|{vPO&PEK@^{hZzrww`91Z$JACud(`)Si~mt0g^HcoVln}42pB6I(R|RMXCTFJ$j=6O6UwwYz%b2^0Bv9rCP2^e-LfIP9A}yEg2M&i)&PzrwJ<kc)cvNGu<OFtHTm@@PJtHJe0<V-{q{{ST;$^fh~$wMXI{1#jp9KJ^o`JZo+faZK=YTqeb|tABzhwl<dL@%!FTu(<%2O&Y;Fyz?1b<JcLOXOP~J447eJhf*`qZ#294ElDWDP+?r_m&7gjid$(qD?Vio0O#t~y;m>XcDX0Y=CLK_#hrLYNh@f5+qKK{};&*#}^w$|6TzqC#pzaV7hf0gMU`8Q@Xay8{JV_V)a)P5p8k*Qbz0sjYlJCd?=qR7BHG<Bi1z1i24(5%7hvdtCy-qSsAqddrLF>RWQ*xAR<!RBvySG<jGw_6>AS-c7xVDd%=^6EWL@GFpE0*f-x%rgAc^An;F;9(VolTUir-&}-~H9Qo65>1i;2`S9iq!Dh5G~QXzJH!Djm+{vTkRm*d7_Z)3fP4cJi!w?FS;W2QD1&X8!q%4Fx7uEpTRj-Z*bf*NRkJ2ckIF_qSvmM0RjFVaz135BrMzxLQC4kF{b$3}%ag)Z?};?gxlR8F;f<o9I;4O(-%ntbaY_p5t%>_Y2zM0Cs<dF)?rfE3WB?3UTWw2_shUf+X4~Gk8`j6tHaq(+!NJt;kvT}Cbik`gQAMp%lXx2AL2J!=d6|(MBRE_+s!&}u{M-rWz=#i<$4LLT^i&a9+wzDE`P>41#fKAnL?GomFoHyV914(AL;(2zE*w6_=v&n~BdJ!~W7X?zD%S`2XX!U%6L0OFCp)g0^5K9pk6fdkdX4)9OG)$D_F%*ICgF5v7<%cJm&yss(DN|Ok#Z*}A^_(f=6Omu5B-UU0euGqxdq}EKynpG=IXm<ov0R`X!IHn+a|T3-Yg?&uL08#kKl|;fD)~(#ltm*hWlK80;FbiIG8N5ct2`s=3(UNcJTzlmWqb5Xu`}$9e)8wEo@0RA)CS+J>T=&3=sydUS9EevRZv>n-$HJSL^uv`P5umV&qeG44)Khr8>BLy^PJmP0{SuP&(Iv`XX^-Dc}mRrN;)O6C(O3LJHruwnk^ru?BzAF-~1|E*$gNw!ChMoHL<5u`6^Ib7W+iiaNsRWN!M#TyTe4ib-Ep(q%DC7<UFjZ2JUPjbZXBRfwgkFkwN!EEiXTI>);2K28lRA5d<kz5<4G8t{zaMwrs_0^11>7OyFv!$bKT9@?LaHZ+R~qyZc2LRC7J;ThLSAGY*Xm<;&F4+bD+LXbfT?2;yT(pFPcDnrY}z)tF>%NB~Pi5kG(0^|w89O)V=b`bqxzOo44_*ZXyevvNjfR6QWU8#j|XMvh>6z^h*L6XiF8NQ<v3SW^@);C7T>jj^Hi;*o1n>tG-`w<*^ZM~qPec3W}JR&5Sz;GIXkH(f@sniDXog{_M56_Q~x#F4AB-rerqMZOu2o-gfsI8)#fhJ<~EtzA#fzyCrn(LrfOc?|V1d+@AcaGJy6^z+ghSVkkz&w;FKZ&qGx|k{2Ur-*9v-jg<ky(<fF5fcrX7CE>@pj+heo)z|X@E`8k_pjo)xbLT1F!`)3p|g`9Ya+nmM1jUb1tC;U^ry>LMWJlw77_DI!cW|%Y|+UptQ=+xfpi<sj}`3vh?aBi`?of{aQg1@^6#M;KqyVC^SyvjAW~hU(*Lv;eoC)JP1%Jg&IpZJU6>t?@G@qfr1&LX-?a%i>%iLmrw(c?0HwC(M|L#b#@ZaeDug7&>2Rks74ik6Ke43%SyABldBRvDT1K$&1_DvyzgeV>Shi{fIFgg?wC6sK#xI>l!gOty{(%jD+;I+Q0w9Mi}#~=26TW=^Xk0uT+5LM;7==pL<N)qRJ2-lGzoVD6y2Z_FcDO<S_)_kEG`U$H~$Jt#Wl{TRknl#va^5$GCxAvgNxi2sHcsXsX%~oWe;h&LNbe*gXS(x27o?vuJq_ssRFJZQwDXwZIAjdwQ{b0VNaBNVQ%K4tg8tmY?bp+Q`p5!4k_ord>KBU_xwX>?i`e4xv54zt~&_om;~sdw2gYr6L>X((`GDd*k}wUVU~Ghis8L-{csS?Q6V6N5w)fP9AI%C1Rx4cM#K*Tu|T_J4wk_lsDv#R8`OnV02yF<C_XOcC|h<kJcfXzkxLdm`W&jYqO`UTFdFQ@Bs}VEbbv>Ka|!GZfU6hU-g(r|gXoY2`Yj$$?JBX5Vz94gFsT9Bm;i8Z;}U|Dk1Qt0mkV9^QVR!^*XdEPvls%-_nzw-Kt?c3RQ~7rgkY3sjD|x`G$wgg-B+h5E}@LN7_kkgG;pY?AMs=;eHB@Z*_qNuQ`A6$*L1;eDXu7HjjOW{mj233pvzUJ3s6|?ph|q6WN{AMky0R=G}~;ASw|ACOTZBS9o=d>&yM0h-;Er&MV%j3g=gYZ0nB{6I-~;KOi0DsRUsMZSpYI#o><Hoh^B3i6b&M?mKFi^TGPy-FE(_=R2{Tdr0fxOarJW(79A;866P&k47~xsJ~2XiASLM=_XxO~?n-LqV;IJXMl^;-Jd(n=I!u#Q0#s|(V1jE=-LS01vc4Fzj~Xdg@oGuCjon2^9BX0Ky$RkA%~DT5DZH0S+cV&MV1w8_F$?f3++`<i<NwaFfeIl`kEm>=#c*Qvi{7A-PEmVoGXPu@YM(~CsH#39wTOXNq^T~(TB0(_l*n~KlFCI_+awGwPm9rf25bf#fz-OuoJbGALy{O&42A?Ws^XWhz9`d?5~M0+|BKFR(oXA?I^Z`D4nZ-c7^Y+Ybr3h#uI>VR1{?PEtI&a~13b7%P=-}pALr9a8~7MfCIE=_`EYdk&0e?jT>Jd`d3w6!pR~#z4Ar)IRz6FvG6N71$H`F7lQKy)ZDS6*2eF2^Fio4&$R*gW7|8+Ru`GfzKN*YsR$-N-(xz!!UIDL^mR`+)eK<%e$Up?Z;B)+#%_hXQyZT1nFREw6KX}i=fqwDgh1OX>!)Tx~lC>f3MuciK_oR0fmgvaK6J_2)3KZ?5Q<`G(!A}ieu%*wVwOI$oa34*9z@^T?7M_clMp0-az(kwL+X>}7^sb6$qoi4y%eFOKsM>@9R#&LkQ;Wj9!n560jX6;HadqN(AdgM#&i2LrcDIk7JHhTG*$F3gMB1Fl#ViX)k%Hs5%a~wnS=m!Tg2x9_4N|C+QeLr&yo~4B!wva4eldjI1-hT8SJGn8X0UL)_=KlLPkBeN=@SWYkb&$o%O^+7VS8d!vnwNYL+)6SUrp&WF$U-sPO2U%j?JDttknqzZ!Q2|3|bORWKpc|YUrqg>=tA9)@V{X(*$ZH;0gi+@l-^IRha{FgRhf7L3F>(E8+1m1!ChkghLkS!9fV;Dp=PH4Keo2*bYFZ?Iy7b%#`%iJflc?f%8N<3rFY6*lA9o*5l!Bqc)!JEt`o|aP@3+v2&!RU;r%8Fgci&bP?2rY!XHDCU&s&u{=-OjDyE&sq+pYm$yp6$fzgW|IfBTOXSwbIA*X$1A{)J<UngpEA`M>fwWetj+=%1lxE(c@~N2<P{c<~ROAaYT6A#T+hhZqW`axA9eN4PsVLr}MHozkLv6_t6mq7B!l~$A(0+fQRX>?>Ap--BngI8t4aSQZ5|tcO`V&&M!j^HC9FZ>)loq2R0c9(-sJ@!dT)K4OnO6Pka3fE%XAQw>)s{uUl8QgS%ye46?g^ZVXpusr+{k|!y<zfkI0JhYLObyajU#P1P&N9?J5$;?3S*&v7cbG?sHGv5ss=r2opoGQe6H^{FRoHpEt&9T6mVJZ|707{6Yg6hTFoexwn<pvwo)o<{Nw@w`YyH$)*80XVnDw0ooirrx&b9;EZ4mllj2lr3t04^P;Q$%r|#Y;YAuJ7sj{Je5KX$ZD<Fz7<R~w0KG*gxbuTQz4yG=~NR<>odi&cb3$9%q!S2Lq?k+(R=W(%QRO+XQ-Xdey!;eU3Xo%O#TRXKoOS|NZJ&-PT@EIm~I&W&8fIctjyfJ`gF@m;t2XtQY_zDmjn27`W**d-ND1x|g*<fI{k$LGm&TDWAR->&ss#sEU?Td!(h6ZMdiHlQkbwi`Gi@|EUdbwl+6w~eO1Tmy3XtH5v+G0N_Ojv!O^t@@j8_!H<5&Vjh6&g3_BRWy9bEQsxnEMTjAtE`nmL<|8v8PKvU8|re#)Og>=<CTLj-CAdWA4n!*0ILAU<OLC6JNP=w_4j<8_yJ$7cd2Y8#{UT9{XGyq<4PMPC<IO@P?W#Th&!x(BvzRZmplMNF$U)7Qh!pC>#C7(}0GHE-~<%_=`eyGfzO30IM9~CV)P(=%h4=Krv^?#G3j9h6F_kk|m%@+r(#(lhsDstvi0&+o%efXFIRdJS*i14WQ@WW>psTv&tpPID<WK+Uofx76i*!NpsZ5wN)*maO&+fT;1C)El9u6-8br*uqW0U>^zw_{bCH3dzPVnknt1pgM&IoJw<|pz?g};5A4ltwEUur1D@Enegb%q2)48BSIgmRN^^+_e7XR`9;ovIDe4$aXrEvo%O;W53Y<pg34S7+=6-^s#+;n5=2TGap_7^>)T-!h(^p&;HSa!9cA{N!tvpo*JM|o_V|#d)n92@4`i<`9MejWQ(?0c#I`uy#lsHu1hQ>rQxwQ?-08F*#DZH6t$?EwHTmZG54gZCnnl#f;-?O-;Yy4#mPj_US{ML$7rFmNc*E`<Z^7h#t0@e^jTGf!4Kaf)GDSuk{{<urhRDK6*xpZ3jar=s5uZ4;Y#4QR>t&<gpg;*u-6g|`9qP2wWKT!5aA^z*|3Y4|$>M&&TQ5<{dm*XF+)9EA#k7aY71nDts$CJ+PSNz)ctoSNrEJdFN^e-UK675&Q-WpTH(q6Z93jUbh5On;f<d=dTd0+rKtYaC4Rn%s3l)vHK)@sPgXAiaob$}nZ)a~}$0uz7*Wc==txfKh5r5x3qLC{(*g}J5qI`*$L4ZyX{(f=R=ad`$l2z{d33m|k3ftUy!`id{5nrgVEZK?fadYWcgJ5@SVa<gfi6^{js@rs;vWVL&h)n7$iyU+y=<wAE8|6IU7{~rH5%U)F$g}JFV=Mq7oI36WLxFF&JYrBBcUn2dxJk{!1je5`b8hPI4?NFFkOp_e_?A4F>8x5Td5zM0aoXKR?c$XQGg1BzvEmnj;5}l*)Lceyg{3~ek@>Lih@Y>+|s^b|bcJay+;8R6dRa0u&UQzsj!BgQ*(Z(KX{Vd9o%H)3G&dWDFG=T#Oc!~1XAep9B6IFmLIdq~FH{R4t*RpyXtKKYPCh;84YU!Sy0IWbm#|H;~mGm6li(oX{=(e{0bVJy7Y>{d|U;|&Q)wk5H?}!~Fj~Ud7f3~dyFk{hA3NQVIJ9l37G_=-P_NT%4TLS})J!q2%#V?ii+B4nm|I6VP$h`V^S;b-wjwQ>gY*QoOI7OQ?%R)86*{Y|#l8WJ@a(}ZOUf@I9n6j3*kk^UNZm$^v($svks8)6FKvVS^LvNjqLd}xP4Z+r+HA@0c3a{N{pr;?ico;ToVV1kwT9G40(rtPvE33RqM%!N7=$}f#N;qF-x7^fZ*=my6zF2#-mshpnHKeDy1FHMUsZ3aYFx4n68mB<e+;G96x(xkAcMWw;ujka_Y^|`<N(Ob!4=ppQOCfu?cf!G5GQz<4P4E19uc`XB2AU*LCrMx@Bs;(aB4}0aY+h>B8Jz{#A8C4iEDN)V-MEmTABFXaWO1*FhIm~KmwB+vuu`T~1($y&eU!bcR6lSq4p{+TSLA|PBX(I((Ad<|PaqcAf=RDdC=t~yHeW9-3_{RT1>-mu!)e*`N$WSIYoY-(JWPlq;bMh{iRt!K>+vLtYCK+#=i*ay0<rM$V;1Eg&Kn1<@{(AhL@JEQw)cqbuvbx?E6sZ#2=16kX}nRHxM87gw|zd{2E;24Pe9r>O1;faoU>8OMO(PV8oLF<+-5-i!i}b|9LT^MZl1BS!d}k>*mV+farV%%HYaJ!Yobv~giTFA4PBdS-zK++MBbp<@KW^v{bMLqy%VP~+YURB>3Hg<B&vhF2cf)6*?^tDa2AHxfnvz%sTp?jct-Swp-hqk7lTBaby_SiGfDoJ0)<+QPY1yaTM#7@C?FJMi+4r4*uo{ksNdr>)_RmO6*JmXkBu^-pW^%_Y>TGkr2J3}dBj|3*>hH;wm~=xNFCTTQiQoOg4D}ZaYpViBwJaemW_}--4xah=%v{h?X|8Eflhl=>L~3xzO^cu!=Wlj3h+TyDRDZgFSuXDMoOh+-AIXPRIVIJQb(_lK8kqq3@=BFRx=$<ECo~XcFc(%O2L*vVR3o3v;a;Mg&K<T1uu|WosLe%Ok}BIHS2jEBMpJ&WNjla##o&=$b7~7*x4P0>K6Wr8McL-ES}Y-6D%Lt{6?MHHHs`;jcr9!`mRk@ufDBT=Ep|aR90(4U1B^vyY+0Dp-gpkRSQ|I!RxV%U1I7|7B5E2R%3~#+Y}=E)$p{kH^>Td_EPP?(yu$DstaYQ%D~97FjX%_)ssoUPw;`?3lu%J<vDO(q{N<wgGQa@Pq=mxp`-(K4pB8K=(75awN&Wy)WtuW<o$$yF0%BjhJ9s~Q9tS3On^sbYnOJH0cI3@0F|JGPW2(Ab_L5A#e1^RdNJ#)!lpgG{ArH=46GP989sA1uUB+|BK<pvNLn}~hdb>Urlmqkc*}CANP?|D;nG)yN}?3YEVd5QP~quH=hmlY)&1iwxek*ooL0MJnhmpj4blk5Bi8#Cw%XQ7Mb`P$?MpiB`|-?5G|xqHB_tBo`H&~@liNU3GZWG+8HVE|F1Qe`v}&dWVty=lHkYrdcy-Vg@O2v4WGZPqCvarXREkT;-TV?DwF<H!OB2rQ4uxXuApabuam2aLP$U31V$dnXgGsaoiN&TNkdAS*NGVw`Wr7w=f@rdWp(1D$9p_UR(=gsA6o@INX@mZ5(%%d8_agnhM1P;bzq#1(2g<rn5HT4`<ewDm-#%y}bI{)oO_uLTZ*n-rY>HDHcLq^v`xrk9jMIeFbbz3jy;m@~V1)BNjIoW-sDQK+nRFR9C(~@yr~AY9I{=k){<&iBinxw&SB^hbsZ~idhJIa8jCNNxfTW!{%|(7Rp$!&~QV@#qX_gqD_i5LmD$ifug#x^+5^Q(s7AVtpQomdm7gp_rBA2P22pj}V&$3#&P^)44*T<M#c}76&`n;@Eph7C>Z3m;}9kD1@yG2R&+~!1<FeAv)F?;tszg0VXr%zX-7?IkFW`b8E!K`>)GZ3BZ7WA59&}0Vl#f}PLByZm)ghWKeuD*@7QuJ9<v_`Pk&p3I45WY5~Rz*$$A!N+`fcq)tD8igOxbyX<767=7v8m4CoLpVj*UxW%HXioQDasFCBKf&~k#`IO9#maQSh7(`4~-g$hV2ujYCvq9YN?}}%E2ZZt<gBjEh!(VQyLPJy6?#(t78CK!>|sblp?0{)KH>cIE;rtHXv>1t=$K2A9){q_k*v^cJF5Ax8VQHyTjv+_U6gD@$bcxR`WAY+#R;8T)xs(0(@#mP0^s7{~HL$oZPqx)Hb@z_)iVjpPH^e_2-a;me9h^34V%Ky4kvoZn7Q$o<7wx<aRU#HqxUo_jF1z41`3Oj}A_VgBV$>OcG!k4wI1hnRGPxOk`Cdo22WXAL%6ga7y8C6BQ|lieKA%X+-g{LyAD69CT8F)^-?W1K5t?43ivHgP2z&<dqYeJd`39`|~E(CBB3{aLg_yiDt>WW$H9|zX3Ml)>-4+A3*>aOrkYRc(w~uxF)-c95ke<D&-Zjt+qOy<UTEdS+At{rYjzxbpoNc8gGd8@x^Uy8Q338lRJa%G|zeG+69?f%jTHe`m+QZ6oBxR{t{St?w1&9F!SuywiZ5SV?+<IvF*=7?E0wOW--H<R-s8<5ZP*W)YLS9x@>NLl_MMYbIQ*`lY8gJE$`-y2j9Bo{n_hRZ~13%UPP?qX+gB4X}2|J+)aCVXr6{C<U>QE3w==h%pPEVMiib>#WYRs1e!u6AnhtBz}v6YG0zz(u9EyG14^~Sp_gj2(~)ATAJw^1>~kr({%8O0swZnldJ1EYLl5HAWH49U<RW<5QNk!3;Cni;H#WOrwuf$xcyb>c9CVB~#(pIw1>G=Gb96xl!VE;EbA7*}nbmdeWmyLD@o;zt65#J))zYR3XW0QCiJw|Kvka>Sj>hC<(xWS%POcNmoDQVOO3=_iFuc7q94dih+SbL1Yqu!3jJ#i4S5Q4=oYys^0J3KA(63w3(?>;QWeL*2f_-7bLK=Z`%)Ve-CO1cE`kBydt1dvuz1d}O(5Q;%KJ|k0-ttUww{TX~dhr{gPaEPt<2xWp@-|R>K{5>9D#4o}Zu(%KgfPj=rMee@+?%S@S?Z;$i&9U<qT*(V@iT_Eq$atfs@5=JHH``xrrPyw)ejK6&gsAs0=VrzDLWi23mv`DaKs&ZJ3w;8Dw!F~_CSqnk{RQTL1thOaXq>K%W((#XQ9h-1RHPAcdLe8!nUVJh@1oWNONg=Nnz;HuwU5-(qB9W?W;2A_7jeLL9rjY1ZJgZy(}0pqf+fYNcWFPn<VX7m1|HOGq>oXN_BhKS{SKs$rOKVZwHR<>UDKBINtc9+6(KaN09-X9hD-+K{ZyNIE`%viXkc}{3_d0H9l^mil-yX`Mg*B7>N3(_%_~<=g`Jb91t(us)vw6DqsNC0rXh1!P(`5S}opV%LyufxMJ@y6>plDjJ+pAV|aKE3304%?9FQ&urYMsk{sqNGuVf+NeY_&$N%kLAAS8_KKj;A{_xfB{h#0dS6okd^@G3q_-Ak9t5(b9XIbWC8zuL4-KdPO!iu>ROI0i=AHn!f`%iU>8syk{l<8w^^}|X(*PoX06fUcygH4$#i&f9k0VOdGC%N?=f7gWcRizZoff-u^5Nf*Xx8#7VDhlk(u%=1ElptWLDU88DsU82RS6}?xnqYn@GZ&+`0jwI4pf7ftf*ECG`+~_!zMo8zhHwWl6)V|Gd~Aj6mGEGr*BqxAsae6s;VZfWjBp6^*t6-2`4s$0R|^bBeHWr0Yc#}8>9o?cK>S6vf7@HmK~m3JHO3s9&yy)YkXXy&&KL|@X~33s6wC#!6azNroX0#g8W)w<ERxTGCE|7LS|zq6s0b7(hzqd2wP{76NI4aU;+<AutFbgW76|Ng#Xw+%Oxb9uZi0C=MH?Xr$`D*L3DH7I`!|BysPsjMU=V;~iVYB$CmEA;fE5A$nRuUnVe`VqCa*r=(Y2#AhKmdbEN=@ax#^28_yhf+yalyk5B|S>_uO5-|C#3Y-GjCM(=ETef1u_KE&q}uAXncqJy;&26`t6QXD&Q@v9BYG=pHt*EbIdEMSja{UtIzR4GHkhE@<%Pc(aPO1CDfeh>B|Rj9w+GK5^E&J2-LJ*6z+8{&Tp7560>m?LnE1XS)}=e3T)AzzH~CsK_WW0^u0I*(;SKOGU7R(fC$<UabmPH7%+xnO;(GMYEW4_KSF(FN%ASYDkKVn@O^_n7a~WRQjr2*|qu1xPP*v=xfnKqN_b}b4ut=wSd*UJzo+n(|v}D%5buz(+>bj$v;Fo!RD{xZb5!e7lWwO*AUyLu4OR2hdEccaiO5tFj&h)gCoq41en}=G7l>HSdcw{Wf91hi#8U8@JpF6+62*<R3eIKN6B>h2n`99=X{dVEo9~=c~fphQIA033Ni=<+Su!j@T?b_s%nMT3QDfFx3Y9f$~Mv*sw-##kqoYQAQukQr~<JxU2w%0)EQ{lDmfa)TAk?RpXHmirmmC+RBBsF^f_ZYw5H$KcOTF<<wqfv&4hntS6Ky2M#;C$g1_)&f^h@Xp5BhunvhwPZ6clBWWTdN^H-f?WP4GeSwEAFa`!Cc#7-@@{#8{G+n{a6<C~SvB&RbohbDO1kJ@7c{03l$kok_dE-<Mq6HE+Y4dsxtHU=aUY*%M&$d(Fw6{)b00G(zlhct-)$*o`Y@`oNIo<ar}X82UJjDhoflr#b?<M8c2Bb6B(AVMvCX)?t0LE_-J(9_Z^<rXR>Ot4WO+v$}qGi?`QN>AY~OYocivIC(a(DZQju>$~7i6YoP<16=l8B~|p(_?)>8978%;6l*;3LR%Lsi%b`g=V&hq~5ZCWGtgg`1!O*ipT?8McH6n7g%gr6v(<Kh14R}YxBAdlI^ux&sBode0;{>Os~sLD3v;9X`PT*YM6Yu6qj2%0vwLv`C9lT;Z+aOfnPFx*_8`lR5WK&0%yG0!Ey17s}6?vQ5!O|=EWh!&Ax3h8{*C%sOassSK>FwX|ACz7|9BsCL|eJyk}1T4N6`M!jpe@ap7XO>to3A0RO;?=SHIuUuZl~LH5e=g97-c;t+-gGLq2{3e*`=1`+g>2S+#w-T~1yXPXdRlG#D0|M@tS+wBVY)ruI&j$_EIJAyO&j$ny**LSI@vXoD1Qyg@m+i;f^0^0zuO)3}RMUOKSOpAu`gL;YZr~^dWx*~tQMovrK7_&8da0k=TWst?<OeJF^Ppx<-B@6BZ7@Gx(S#(sOFJanII|QfS7%!+6JDUZ@=$fz?Ul|A38VeE>!CDAS{`Lo7d+_5AzWd;(4}SIGJ0Je?gTHw27ax4*!M7fK|HHRGe6O$kINX5uV+*iJA<SaHd;)=hm{cpe88WKIQ@CJ?h$&9%W|wA6S2TLgGk8IQ*ikF!F2>3{LFj@DW;Ui`0cw;(mUUIPVTe3-2_;UgX_<ypy$xqkLxH9WUhG=_d}wOM+g<Rw;W4sVXnhuR0grUZ0FGYC<mQa0u`aAAx&~F5@?t_~p!%>r8(ba|0!`Pb2La0+rif1Dqps4FuTBnx|7~b8Sdaoh-)?uK4uHrG*&@U12nz^>_+m>NbRGz4>T1$77TVOksA<#KNj$JOwS7@C4q`NWAlft@wm2{a<lzGgf#E^mHoAxOE7>7Ob#nVd{Yy4Y_9D65tj#y~VR~$FJ%u=ih$28uUjp7b0uO@s{f~bA%|}1`_M_kY?$LL@_2?Jx{{FXbBMpD_^<V$~{trI-)?a_}-Zww~&R>4~{ePmj-@gCx_y6wE-~7YJ@Bi-6*Wdc&SHF4mU;g2f_kN7uKKkK*{^*B)_sLuDJ^J2PNy+`czWedde)Q<Q-+uCqpH|r>|M$Q7<)gQL@aRYX@aPvmB+&o;tw-PZ^~eAG!%u$ot4DwN^+&(>4Zi*0JLoXx&BhPk|M?u;#~$AQnz!@dA7_xhegUt@we{Ng!O!7E_TU{@jGc$?znged_+~6Jt|JdwEf%d^@k-`{_RfX6W!p(W1)9==RPjiiFju_J#Y=S%FzqrI#E`H!u@IRK2b%`lwYdz4HgqO9P}o#TA|<E;1lrADhTi-^GDvOIvv!1=FRKMxSH9ZM)95OZIX1>Ep^CK~0YI_^q{1nEAO_9i^U1hh;{01H5ns5X=u&q%=`%$vo2x4W7rRAC+_zy;86u|vh8Y3N=w4P+&G~XX0D&tR<u~whN(k|}emI|i>N34I&5||pl%}VgAFn)dcN^M@NB6GKi|6lM0g?TAIod_rSvTf{9L15;;U-(mVFQE;DLB!@9AiULy3WM^Qu8xgkm%<g<kz`}Y%E;h#hGvJy@B`Zgx)kkVt)^-%t*B?gC5@d!#n@u4}bgP-~a1dkAD6`piuJrqrds-N8kM4kG}ECdpX9od3)mrze39T@c#FJ5<GnWo%o)5_23_;KvW;T|C1w+%C`LK+*v&qhFEcXG7)Sf1VljnI|p_eQxCqwcpvdmvLHA4@d7S;yGMC1kx;{HxY!$S%!IXwX5?=FMz|mHDM7#>L}?F%@_T>y>9;?5@2elb{ndMea1KG7p}icn>%k*{|3Q@r9Q(WKIMsKp74m~tK@le}d1gdXDA^fF2jPH?n~SP%HS8EYmO9?xE9e@Q&8H-wZ0V@JlB~4FkWr@x8&Df+W+gSF;`Ers+tLhTkqZn_N+|M{$x~p+g-Ncr6qzrzU{Pih5Ylu}zZMs&N?16Z#u5}R*~wa9Dq;zYqm9810O_kBJ0U0RO-XMpE32;&#XF&~j4#P6?5w`!T)lcueMAT3bXbqNG~cMD^_PfB=~9Oyh>QD2t5+goemuG@vu9n!X?-VQSto;(ERd=qn6mm9Pp<)C06k=SG@Rtj)Q&o<BSBp?wPWcbNqskHPqk_!{!<S|jz|7V9q5V9?Wh{IZrz$%4kL9Ww5!_csEQek>4asp{Ha|-y5#!NRjwU<(z~s~@L14*8`**RLa(v)F%We!1|nDD5s4ICErq@+&qNeiCcqbW8mrx5s)l1KI%Fag&8dWvSK@H(u}QFDk)TO36zEKxR>VMkwl61CFvP0c(gQ31K@TFxtGT64w!C9_apFtv!B(RR8E>N#b$Cxm!pg~TbcW8ld%KrkmuwRPeTN#pGKn1B=qxmClI#YY61rEuB3gg)F`Tih*&3P_{jq$DWC75+I<k6gWKr<+3nTefm*O^ReNBtHx7EJe*>%Uc?*1v4Yoo?e)!Z)K%-vr$p4X@-Us4Q*Bb>xT)qP2Pk<67lHr`*jRhn<Q*G?|EYZZ4j${Re?t=sI&*JRo$-K!U0^1Y}v7g^d@qt@I}$(N$Kubo5yCo1HYk2g83(7?@4P0Bt^>!+?k#b<q|r-!GiIZ?7?Lu5O1x{T1LTVoQ;wg1KIw_m&Q$}K!!S)O;?o*o4B9MBE+*rd9co!hTEFB$V$>YI<Z)dgX3X<x&AAEz#2Wk)%GUY*_6Wt$El^COU=W<fE|>?rHVFJ*1wAS#rKDRz>}sGlyFAQOt)%<+6n?0%rHEWCT90raNPjx^1Jk-^rELO}Ba4s^AR)8f3-I^lH}zCtz^b5(M;C~d@szN!ySb9IqB@!7==#&5I})#5MPcyy9LWRBb-D|_#<hu4OOaK0U2;sB2iER6JmqfkOWFwP5N!e;0xsrITtbD5V>T~J&3fRxjQ;{c^l=g1klpa-VTNm=;XbT-C`I;y9*P)n)J%4=a33vE8O0L@d*5a@;JSOc;QT_sq|`|^XLVMV8^+Ilrg*I<VGWOXl=r>Ig5i^sy;7G3<V8YbNye<LPGf23b4T>1(+Cp}6{66!94R~=!|UT2GLX)TAWko3q%CtWiKr*;W5tBc0e>`c`JC(Ej@DNharu{gKkZTbgp7y^_SgKY)e8l;9kz=$$|+7lE?2Sx%rjV|)Ey=4|;6q0cGwr9(GQC7TBCZubm*-VjraXmJ)|Kc6GiuF~9TDyK24HmG)n%E|$Fu?>3)Cu{)E{sspkdzm6mjGfpe&}S2ohb%VD`5;%6_uqib0^4P?W9N$0vjE+1vX_Hoh_yjrGyHR_Oh;h*p$X$T;*Il2)!o&I_wTqzUY;wa+{D?vAcb|J#C{Dg=Nn)R$#O&$=vKypv)~xAhB{m@v#7@0+D5P0=V6GxMy2^#+_*J=*lTy<fa0+DeZ6yE_S`_*12udKGW6N`k?(L#InZ=DXtYtgAuS7W9@x^OY|P6&P8=wQ;2qx==pFlooA3dgOdNCzz)Poiv>R2)1wKRam$aE!&h)!F0=vGBFMeD{Ms?!D}k;kacWWROyHE}r)Fx~IIk-K<e4-8@e73o#caU6)NrW?a+l=jDQ<gsquC-Gvomw$dUD+I-czPL12<86iWuN8au*g_x)K^_LJ^>`7yL-}RsNMNMPhtRFE(6E(DH9Ja0XmZ{>)67{W8ITj}>bDILl}KK5@(@D9?Z|L}`TV37uOY@^Y9DUL~XDE==<9SyE#h<{&5mW0{g@2}~AcFKreIGYF!6<u$%B7glp01iK6siy$S@J7mr|wkyU|w{)WzI4r{vG7ZU{87^0*%TbU`BgHg4gTa&5Ku74BdMr8}fcxRpfs|p@FHS1rILXc451-y)hk9TZho%h_-B{=qr2l{{3S=!QX^7-IF^UiwP`<oRKGW<PXNJ7aqG+$#wX{?w_fEzc2#UV2S@I+iXT52*KPX+p>P*bW<!A!Pe^2^^sjw0}$L0IYmx5a_+`RS@9)PZVf08P#Hd7k2M7Mz_08vtyqN}228ZcY#QEo5tmH-U749YP&82EC|&|{SddE8Es2d`!~tFlls7o))}U*%>hEC$PgsLAutmUlN_x%N_U{Xass6yx`;7yix8L}hqKaWok&zpbwZ?aqRFXTzu0Z#nVFnF#ijrFs39N+^RRFjAtdy`pWc36CaGhG<mO5Ksk`Q$Rj7<q}XSoDo-Xu`Wspvl7dQC6V%wdECQKO_==urT03>Z247RZcml$^xIC!m9C$!6J6&u(pqXQ!MuX@X(6B%@QDjCq*MN~vUqx%jYBtHT!$O?mI<b5St#9d-q+1$8#1I6cOf^GrNx$62^ZKB17VFMT62Nw`Yj#A=-jm=O&4<#XF*vc^K^2qG6-hpNfp61dhTwjd=x8IHtCdUmRo6BF)W;Mt8PjW$+R@hpOiHfaW7?QQE5$#6{yHxmLq4sVR!=9gECDWmK??PDe#D+tVxYa38FEiw@3m+@N_+k6G=_c$(}6+yj`AB?aL}lC-X6hLyqU&J}y~C$EX@j25`3<aiJ8E)`TD8W=#4oa=qjR&8&*#OFcvn9&e+6>TsY#Iu0jLEmC4tZd2gnfXjm-38Ph+5VnpBaUm+EC>wxZP+*V{w|3E?R<$(j+xY@eNLF6gQ7;@-0ChlL1CCyiE`1h6d%{9A2YfwK#Ry~#x$w?WfL_%$ZDnU5$ZM0#7ZWz+A*JQ%m6HdImQauF_UQR6%W3*ZXZ5JgEikI{bD?)ZOSW!Ajt{G??Q#tnM3J*nz;Ef1_{C5#9##LKJc|=n5uo#dSBPpNhNamO3C!4~97tNfi;Zbm3bFA2W`L_um{nmLf*_F(rGqnjZGuwCN)MuVcg&;<`*gT%PG=udFT<Mx3jyumx1QF1!GI`a101kQproZ&9I-@b9%^IqaaL??=&0c>(sz)FmZ_a2Lyxh(<hM--)pD&gG>q?+Z-rG}yuP_i#UGM1ESraivS`XvpsIdpYh9PUEgUdS)WW8VA+{|np~N$az*NDA%g^c9Bn40Hm-hQv1*8-`6)0CzJA5VdL_iKmWo3m0X;dOc_-jPJYV}dBHY}&4YIhly?YY!06tXQB+By*cJ_W^5$LE?_aUtT>Y6=(V;)0-r{TfkV$C<mK0!3~4zyb$QG%b*Go7O=o#nfh@Y;&OWSWX98MZ|Qq19WbaZzcrw4&!hpu4ELS4#o(2*Axn&xCo*as{x%qcZuf&>ul3SG=nIfG&|2-0)j??90D8n{ziYr7C`U;918TVQ`F>gp^XxD08NFcu0q3MoKY5Da$M!PfvWO!Qscm(aFxZ-jzmEV<x{@sw<7>QTnu7#M&wRG6hQ>R^bmX!0sgVb1quHsxXPP@uE`UFe0EvVrDD!>YM(tGC7Yj0utzYp%w>&VSdNBuVHka|t+h31WZESYxE$KPvmzxERmIq4nQ!>`Kb3hYXS)Pou}Xyc7z~UuhiWhm@dUe$Vawwr#`_F(QCB){9J}UiAWJzDm_2drU<-Rp`Fv5vM8^sx(G@;}6?L$ciQE*ZxxJE}48AEl5iq$F*@iw}J@yq*Cn=~2vY;E}NwBHDfGT!I`UCqzEsuix8uYaRnAvtB+@@k*9|}OmpD2}|fKuSI=Np0?bH7}yErThAgZWQ-FFkz!cbFUx0uETSBf0Uv7{3rFFW<OLk#L|wI=ALs8$Z1N)(m7=12a!N5X~>cmU-|tIU@fgm+yqeCzrj!gMY@Ee0cwF;e#^pY0d&P5a&u$fGyD_nZQMuRWxVO2dH`a;O7t~akwyAERGN?@bLaWQJ}?-X6U{E#pWD-q57}szq<$59=`wgGw%=(VS?V(W=DjgVK-&~KlX+TXk+K${r?0Cl4@Z&_-jUb=i&YT2~FJpE4!0GKv3lY@o4YT<k7Tipfv|uf2*_kl)IT*;|KpT#aa7}WZYG=AcrT6W&h#(KLe38et7>qUPy0%IC0}eZ};K-?+z+peS-S#tk4Wh9=u&7g@=_NfB4G~bhB{f5Bj73&f5dnU&CJD+lTl69&iIZ);r!y?72RNLhr^fW-dnN5D;a2$8cT+&>y}-bsoGkFJ`EuY-0=~WROXQh?<`*l=nQK2doybEbs;6)&XS(;5iIkMUV+NwIEW9D0{A5O$+7U^ZrA$Lm30_;Y$8B0L5jy=e<nr-1ELr><cVnVlDi>s%GvU?y`H{+S-~&f47$Z;}-&|$we4Zhq%ENp&XsN=N%swg!;te-=^fE-W9l{-W0Ndeq|_R)T=_~fj8$2Duek;uoK%9I{f65Fb6Q{lpglMJFyTK|I1(gvYgt6u~4JC%@X(9WGI~Fz0DR^_1n&jT^_Y}4JZ%qEx4Wg!(TqU|L$yewPs5Z&F*HEHmK^k?L(g6n>X(Kd2svstFPR?)0xt_q>~76aBM}VSHt39u+0dWU|UBH7e4vm8%J&!C69U0j;r!6S3bAey~@dvO;H@(%C&9pwyKh-RukmX_qcR4PVNr;;*@!yY+<h4T_Wpg-CaW3BE?^pE#qXpwN1I&%)vHO)6xMZ(`#u&m!FYeoVfk5rHS3V;8dmSY~>zO^1CUn_TuXk(Yh}2jW=G@c36JXu9#C~aluLh9d%e)V2tqW(B!*x?8*?V;;v*MlATBU_$eK+(}E*iU&TYXDEes`OBi5QZ!^KUnC`(y!2K|#<VJ;sp7P#-RbDRIFuT;Qu`wAjK;uOoPdb=$62BK?Lx^pW1n}9OI>kV8Z*UpO@t$j=Yg97X$14`w1<F~z@A_?E`RVo?Ub=MzrA$ZEZ@pl~I?2^*E<ZkF0w}v<rBd6oFO*cubGoY#wNj!~-zw}xTF7V`*1OEeSWt^XD;w~XJ2mC*<7^lr7MUXuqgku7J4tq${&`vcO7aWaGKOxprolnR%362{<sdTxnQzQw8?K-khdB%;Gy~cp2;XsK)L3o~L%+(Nrw6L~P*}gpS#&9op=33K(zcDP;kLF>1d%z^<u){jL#UX~q`%m1Iqj~x<w*>A%g4i|JG}QCWC0BuF<!5?=kSUuZ+If@46!l+R+Q8-1?bl57XyB3VNlhKD$bCo`eN?zC|9DEC2>m*BkjNWgg14=JbMBh$LJp?A<k(XawW*Ax#Dq209idIoEl74c8%^8QgJoP%|7v{*L3PN92XW>y&K3KjSpWtY-^btp7Ov$9uGw%b--~70J@Y&>RnBDiNAf7zBDmyG@ie}@AU#Qzy`|dvV*2OD%A;x5T&-*;a?e6f>e!|W)ZN^Z|lf!gWAl=ytG3#xf*0UfH8!rigKRQw@|v%^w-vyJo-AgsfU>i!*#m!BoHPX06_FfUJC;Qm)sqyHV0|OFjN~;>BbmhteV9bzS>F=-2qswq_maQDly(A1k#UHZ5&PJJ^uz~rpz$14K5|g@`8Q#mjX-X4p<Q4D6ljXk8XeAyYOlurV&^X*MNcZ6{OqvMuI=_HmwDP|Bn_EI(9jZeoe@r2cXfHO|I%fNl>>0PUwM5PX<o)vq4WVpdLj?efMcBu7no2*SB`5I2ugqfCE}+*6P$Eg^mw%wRWL);)$JHUYQl6yeu#BJ9IJz_d&S!(`fjVHYDzEO@;Q=jPO$Z5CssjOQ%4=n7>n@;B}@FLFk6^-04unvayDthG*{z1f#5hjA+byJ=&UnhV{716#8+Hx7s4ZTG|#LF{cny+1D#=nX`6~J0Q|fO3^`bT+lxVJ8NoTtQ19D?9UBK)q*@wsfgN)RG{l9VC)<QVmJ<RM<V9GrLlE=i?#|aR)NKr<zDw0?vvdRU-`wrIyhqOEec~%ao@z!1OX-6352CV-`Ol5qP&jBH^v?Ul2(PehRWaBY&HHbF!+oj',
 'diagnostics_snapshot.py': 'c-qZe>u==Nb^orv;zmZn;d(@^w%NoOJ%G*b8rjB=^?F^SQ3aYqa%O08$O#{k#_9kq{2}Ov0$sF05d=oPF$~v9gC=#0qN@-6ASeHa`j_-~&b^nsBxgplV;B{%EAl?hJ@=e*&+}57o16Eu&?`dWiG1qCabKjPQ5+?q2qJHsq<Inf;>V93i&2(NG04xn(^zC>(ic(U$7K*DV=?oxBJ#L7cQ!XSH=^k*%?jaVV|3-=22b|WI1c?HN|Riu0r%4~DZ;ETf^g)OaS=p*p(p$KiJTVzy&{^1dXj$c3;O?qGzs;TNimJ}bCgz3{40)z&NM7M%=UE8`+1tkHD+Eh!3e#?W4!7?v)C&}X*SjG`8?N;w7!1+AexP$INaFSz+MDs8n^_$N09sC)XNf3Dz^t40zXAImk;!lg+-YqjEobfUXb$~hd~tKS<wp<KP5WvTV*lYdd0GBo*SO|;j9q7H{X7kWhsdG=U!Zf{A>#^r~Y~ojez6LW?>TaM%Fp6l6cc_F%ah%ys+#xPUnmEe&XdDibf(y3nrrzo<(_)_w3FN50AJdOHXsLuis9iVj}0IK&-o0+DTqu{b>yk><~#g9g^g*tVW3l(GX@3jh-Cc7dr+93P&33pg)@RTAdxPm_LNECY-@0f@a&D8Hz78H#*EPf7UbRTyC?)Aw^F@OYsMX#U0LzFz-=gUyPzK4m5KlK|t~-b>7D2lkuZBdr5do`0rZ^=4lq;Bw`RiIS+6S-{ppinhcSU%J(JkKK4{(UheUc-nSxXi!5wFbNJT;vI9V}gIEU?iDQJRA)v+#^z_b;!}&lSilCg%^4<y26N@H!nT4*G`%$$2su$;>O`MJLD9H;i@k2cj?8ysg7Hq|~MTmiddzNyc5-Y6Qm*W(Q4)B>mnrRVqQmfqmqQMJECTAQLJh^8(EXiR{%JEhnjX@g8ET|3ez|G6))XV0*v;>+-dfGWu;YpYjbX1Jfhrwc7SaCYeEqaKOlhpTEsX9EuGQcSALp$?<Bf1;(SKzTP+9ojXS=@v(x4$neo@WU!3F>|v(To>UW7ry4N2Usp%XQBFxi$3iD7OZ}DiiO-i(+pWhd|cCq>!3_G`ai_*b@Kp3xVB<VsdryQ6$EZmk9sr;xmt+hr={I7SpSXUqrj&SWd{xxyU?GTwOf##h1^nE`CnWzZN+f#A%uxL#nJrO$ty#;&`~VqcE>%sQV5Ly1G%Ib~ww@aTeywXt{<%de#c|v@}#j>LvnA?z&0h19kB=_=;IAuHIlN0Hf7vw&tOq%H^q@NO@?5u{X=Zz*T*UYIO3J&Fewr#a3m_9kRnwo|38cik@PFS6D~3?G$O@#Tu^CBrwZiPTbT4CL{>aicNgx&n4^qbUKT}A`CD=&Xjwv+TCIj<!-%iFzU8CK>j2Q%D6MNYS&ts+^BtHrR{3Bz^ZO0+Ut%3aa*GK2Dvfjd8sH?9~`;3%ZJXu)IEjY<`NC*I5n9$FgV$-K{9o*XOMRw3{qqMC`zIn8$$;UTihe|+GuYxcL4b*VXxGCU;L?UEKrlxE8N%v)|xArSakwyO~gS>Xai80#kmWu<3eSYtSIENOzd0Yd$(y}mUU{VNNMg>BFI1ht%9%v{?Mdr0l;bO?0V{*^{{j;8e8?Q><aKs4Val;jgE5Gm9)^(CACg`YAUr!m^QQ`J^#*E$}#@KZ?7&sDU^RczIyRlB9h6~i%*MPsR<|P<;R#qEh4}~Xw2;LSBccM(bbFpN#r_?c=+_5@B#a8MSS_OAh^Hq75rrESQ5?JF@%5Y!yae;BsvLuFqUwKVb=OWH7R3;u;4NHA7&>YMuDU5>$y|tCUH7}^DN$tl7PI!VHSm>cn+jtXhA!{J|%@32t_!;(DvFZXvKncu$PR(-YbS<*Kz17>>RarPb1kFCePqT%b|N5W>8Q$PSR?x4(x_*08&{GeUR@{_*gU=N0WGLGv9%1?=Ix-4*o&-jBP;me!uG{W?qyx{T9}kc^rnb9u7wDwsZR~PQx8>>y`!u_2s-`YnNkKMsZLvi9L=cd(0*%F$#ca7*0MvHpoPZd%srPc<t#(%qa_Bd4S5w)t>&;OQMlf*>!lNtflT_BiVIfJUUuS9U<AZ!nTHIGX$KhxX)qjVAWyD2cl&!C5VO6zJ8jKHp{}Fl)y)#-`889S>&Dbw3Jk6Pa~pcTDCKe(_zoLCAVTX!XO&b*nTUR+A!pZ1Vzqh3Ir4pPJ-dbl8WJR%9@i2S5j6HfOcz)Aw6h|@9b9t4rQ;T5KK7@IBKqYO$7Pq!S*{(UK7a}dM1L4_TtxN<)Rgrzey(I<nrURscI~H{mw|GBD^5>aWr6T0q?eU*!es)AkjAZY&_^!kWRgbq>79i{0P-tM1}M~VE~IVr>99#%-s6*C#4rh#k^{1uAzHJu{Y-7{=~~%pHdq1#?g^q@X@!l4(OQCe#02IR?xPXyC7oTsY@i+&-ui=bN73^6)aZ0HerOx7%GScdA<+(Yiv);__C%3C7KYcF}x}F7&^N#wc6{DB0N~#8oX|4(OEaEnbmUkqAd!Cnwdv@jB1Hi&8$q93w3i`7qG%#F_7TL$VZrK=p9<55A3BNf(R4V!;nMP00C!Z3WWhE{i3&0$-t5Az+a`5lZ;Ipi|9di%egdEoIb{CUAvK7;dnvDHLJEiXjx3zr=gx(?0lLG=)*Y4W6aQ(pJIJvrNApww&Y|=NwF7I5H9ybI+=j($KG@pc%<Q3-AfGYW+VXrIB!^nW<bXfr2x{F$oc5TTFNrXSW7`sy0@{8G9S<sV>U<K_>q@u1Z`@!tf#d$fuAhoEIFp7q2yRP=rbDHb|jX~l-kQ~^cf;(ldsBY7P*Dt+?7X&Jb*fB+f$j`PLf9ws1}xL{$aIRt)49#rY8?$KE0si%ll%dlQoi*8W;zslv$GNK(}Tb)<KH7lraD~o!)9n10|#~j^X`4ofrUAT8AX7cId}p!pTV$awU{ks$yLVuaG)-$&#WI2cC_p5f4Ox%T~hWoO=#W4C5-FNtyl55W5W_oq?@u0W?O~jVMZSWxy&;vXv08i8wHn9fyqS*r*Qmwk%52v8IB`!4jvF<&<XN3hGP!O3|1pd%Xs~mh)+eB-tiLO)lSLo%M2Y-Fk*vyUAjv7L+R`Q0+J0mOSjO6{vBUE}|yxBgMQ<Z;%}`S5s;Zr_nDdxXYCUsCIKG2UN&~lExuhMniFcdaO+wmQbsqWm`1>R!RVX&gGNZt|YG{mF(zi-BEk<)pVWqwHF3(nYaby>od6Q5W&mW4Va`KO>y>-QKdlt?F4S&1R{^X`^D$vAE%ekfuM4rXK4K-5+2HJsB(Gv1R>$+HhFz`n~Sr{=RQaKzeyMXLVO*siv*5#B!bcm-C3n_5}O>1Z^K#sy6$q*DdUEQi`7}>8vlB!PpxF^#vZn967XfB;5Y7Me;a<bt&-YdQINGVN-avRmla#6>>J=r@86Nr<w>Btxtw#T4~aCl=?M}(QAqiM!&>b3z5~@z%x582Y!CM6f23AX_1ip&HV^iwPAf<}^03(7<N^v>Sp*O6n+HUuyY{wamq9~oFE8e38m7Vge1y1iuyc2IMzvcU>#Z`{1+HM(;LdkvXS-Af+nR*Yc!FWh-9<sAiSr>;OPFm@)tW?VpdWTmqo9}!c5dJPlLdnJB2Xg-JG4L=M}c@L2!h?27m)E9d><>2^lU-=$H^cLN5w9vg1WjFt2UV31>g{|Msa$&b((pzftSorCt(&Yz@c-b$BJ+(ho>G6@XjL^isa{hnL&LGW@$wFe6hDJ_im75WbI9MmXS{U?_d9;I4Al$VeWe~&JD5mVFM24G=a<xw4xd`XSEy}d*O&58VSd9(=%9XpBpa3VizT!e`mdQ@sS*;J%xp{w>@ieb$0MjJHniu{PNl5C#>?G&qbv{c7^tUc?=YfIF7C^KAa*U$c3Wc$@~LUB42Hwk`ro5KQCmalyeu#wrqn)JorO+yTfu^X=Fn8a{%K_4k<s2;j=PdhVqwxfBDNfqmpYi53)RubT;?PX9_zsri9ThWoZ8m)#|H@pO%asjr>q1PiC}(Pkl5&^Al=6i-1%l`H2edm(M(jA869WW^(7iOe(KV8-x3I4)!2x2bOqQn7lR=g*@xm=m{J1Eg+JGT~sp0m6tsWry^#uY~%OEKhba6I`Z2eF*1$Gd`6)YGW?qyJbC~fFyY6i8DLP(X>$2Is#@Q=Up?L@Ia5nijYwU|=WRe#3mvsX59Pnqp$C_GbqP*Kxg8I*zdBs&FDXV_;;)3|ypO|tozn%n_N-k&{83)wp|YGucC-8^^}S36SPCPR<g3FXt*dnmZxM9~xTVm60w{um#<{0;WwjhAZEo76H893vvQ=etNhc)-1k`RA2izDbXu(73m3$3g0irlR*i$<0;3*EkGqCme@>k-|-hT59lHV7f$pi2iG@cqJ1-9RiS83tMBa=e7w_QQsX8WTv1LUwxDa|~xEbAq)L|sDbY5j--$m>@Z|193Cc0sV-L%>_|aU4hNvL%<FCrW{kfaB=?47&h76K7*fyr=NyT=9E@Ms^g}n@myei?{Cgxw4HG?Y|mD?`<yITvH3;HFdHC7d^0~z7wV9LJ$63gk-MLtaSb+p?}P_rK9}7H-@MVhJwf^i!$g$mF+Bip94oY+vKGVL6t;|Nnfe6jUvz0i~k~hMw>6mvt^*l3Mil|z3fl^@LR8imLN8)Ql)CgDmx`%wZ^Jm!nh!(=Z8<9J{IzXQqEc_BQC!^dZ6H`B%T83Pf8wx-*bRW#DuG6%({A3n4CmenovZf`TclaOi-8=K<K9upT8-^1SW;E<#78WD$z-r6#yzu$RtS7q1z?ffKk~LWdQmzOrYyId@+px)pz-FWgmje{|0SY1lbPf1d7Co=;DdHOBl5+2|@78Zpg%QRJx8Y|7A+NL%=~?#H3x8cnRKZ6nG>N(_vb)mfg8cfPbe&48P?lr#Qogro29diRKjXdh`PZs3n;|>gD=mTs|e5{;Q(t@DhUI%m2QB!GH0e$!@Dr8e9MOpM#G$`4xfbSR+r<w6L@?-SwbVsg~+f`6CEPdT|3N3Q<dniDwh-rVtY5ITyXwW`tjIgWN7)+RnxX`zc(MRfq2zb6CU^%#a{>^HM`4<I7)S>tOAKxB|3xHzY!=eN`z-PHQs*qTY<c-h(PjiJ7(L*%*2$RMW*jOo>CEDEAMze&KC%!90d*0huN-RqC!PC$a28vwk6cpvKJ~-d;liI!wEo3d95HXH*A9b(&6MO{QkJP2AH}En2)*L(Ycb@eXe6@($WQ?*@zyn<G_ob#R52RL1leN~b4Z^8Ke7UxuT|)lHB2RRv_a#0@*9A<!L5&MxXYE?Y1%$4)VwDQj$~C(2z~GuBO%kD2EVY#bH7DXFKf!=gh!I{JY#nTS!2y)kEU&KC=#Y|70t%RifYa7j+haF&DY%){x1;>|pIoz17^EEsW5jH>3E!oGq0VU5Ajp?ai3O7|MD-c?VHvfCB^B~u>Dp>L$ct8@{BZvuQ}RH}MRV;_7IR$U`@ejLqSrP_{hpTZ(~QOcCA(a@F4YV4!O?t_P~zIN~F!v}m{b_mSOq&p|xxlP`$B(KeS^6=h+*B?4lN-9Xtr+H=3YF%yZ0BATq#H7DWl%uG7?5_yX^o9UXI&NPPhM{AKT=l(JUD;PxW3K^YxHJtgs{N*53@4@mMrXEHWZ&%TcUgNht!>JV4&)nF)a>?680G2I*Ep&o$t2A~Dg!B!91iX-8?d&@CWkDR!F~C<B3Y~2RxNh)O(7z+b<o{8Wd?_3fY1tkFpjtI-<|KOU(M@4{D}k%Az5YQ$PkP*>ASA5wdUGL>&^+DN1OG=7JPs<*WEO&Jy;cc>4I$?$KJZdb=t+cg7Au3cJn^#Dn>2858t^2(j)_?XF-b_QJ@3E_wta9TYIxQSXYhe?Wmj9Rs&kCNBxzz$`y*P76m|12@=jQx0@GBiCAylp%rnv9A-a{J_yC$zA1dmNos13o&gI=Y*TyxijdnK8rsXb1VSEmwnny%Jdy4v5sf>*D*@%l(aeREllmuEx}D5%h%1~sF5F9N?X~Ip>cfgpIrt#zOBK4xq;FKfvZgUE$SvPA-{qt{Cvb$90zb+v()t3JcWN0BWEE45m)A4~D}_YEqeG4vCV0v>)Ic2%7g^IhI$MEaB`^ve4xFijR%2lB!j#ID9IZL4sM3|oT1~EBwke+8X<y&!%eN`}r1xU41!Cyrt^Fkkt(|O=$U)k^ce6FW9=%qbgf7(Xai}(_j>M8X9yeHJ%?i36i+U3TFOPzdOS4kXr`f!_oU)SDbt1sN)^{<oCoShi_0--AeHFj~Kxz~puXT@>B^@x|EOAM+uSs2hMc=cX;b|^u^!`!H^4D(^zi}v~f>>8)DV*7y9X-kcwHswdsNHHLhniih#S|a4L}^9_5ZC&LVH={%EV#&(Bwfs)z6#8eWm2PSiF$0<dfM_b)z4P(8XenEG1Bx|oIfK4!_}@1g;!nk<$P54mhT}7f=7du`2Dd~Tj8v7W`uQh_ja9dv=4xk7Szs@Q8ca^^R&!xg&$1ea8fsG9|u?EK?BviMkd~R=iY0Np8kdV<l*BtpFEX;rIy2HU}xQX=fR_=&4K#suJ)l_vtTuaf*vRizH36MLPOR&6ljPas}se-Z(01RY~^YZG!D5uP%XZxRzwvAU(Bwy*gTw#S!>8i&K3gqd6#C@y@9TNt1lnmn~9mcS#bSQdJ;4Qe?rg(hR7cup`#vuJt%$Qq9tKvi7%oMXN;mVeWOQrz5DP7?!zZf-hA>li{4s_sH+wyL}jGuqH?N?mUWEe*w{VsQa315pIiT*iO@^pNBBO3N)RZ{MPs_nBjiPu#uYE>&q>mTl+X(4+q43I1VE;bm3hydRZ*)9=ejua<uQD{gMu_ma79WTu^HSpnkpHrKBrhtjs{)TIyVaem(rYl^{=a_SFop!t<EKaNCMX%Otj)EWcfJ=p$tXK;_!#0mAz>@1NB5onACFCRv;7jQDKn5r_J#7`dHq-qBhs8RdAPmw9*(>s|~)c9JfCAx3b~#Py9P>ru@D-55r^q@!c|hZJF${?HlonT{kY_H0)|$uSsc_o^AeZglj)}Qy0W=;$kDgTYr64Cb)Bijc9M%OyKJe%B4b6-*1ru(<J3TsgZGdU;cRw=Uz7Ei<XbMrx&2`jX!a~pXIxB3)6LNV<ts7KweJiSzB9>8cTlh1S1+4sH_;TcDZ~G29gk~sr-v3>#ug}+tJNW(rAdIK0>2!j7&npo?&6blRmdJ3WLV*)^+6SN}6%eyKybK8lanw7W~B=WZC6<iR<zw)2>VOcU^0sK4c?iZ2TX}iL1x',
 'export_snapshot.py': 'c-oCtOK;;g5WXAme_-7MDa2NSpt}fM_)wrJ(4yD|Xto#EAQ*8Jb62mB)Fy6_|K1srlI?gmtqwNDdGec?kDAS9*H2w*JyWb`>rNGZ&VK!Q(OGa{znJD8w{6$EAK7Psj#+r<!FlGbD!?(*;0`Kicg`Gi$|?`lE}zY2(}}60Uwe=H4#thGa@)$>jqjga+e{~=ZEMykPa>8iH~2c8jEdb(S=&A`<xuhUZjM1PH6Nr8WHp^=C^0>yC55O2dC!*93B%`m=Df{aR~cWK2As&4s$9#$j$<FsE-x;x&Oe@ATwJ}wQ#zx8eNmOs?6*Jv_&~-44elTsd35<`{5G5*Z#!rNG)1dXX~lbAF0Ob+nV}u1I^{r*1>f6-ttrF2YL#|^^2u1)T1G2^I9f4NhMk;(cOu*D4_PbIh~a1fZOc|8^7&8pzOSM2H$fsaxB@+R(>5z9bz4Z89h&A!YpIfr;EP4udp-}PHx~5j*7lGLP|IPYWEcL(65)(;v_vu0QP6gd?@8j?-WGtug94HEIpeY6a#a-2dC++f@?-~39Td2Wmd|$0jpmyrvjHv}unK-(qB0`Zw0?+1h*Dvahm3u<3L8<J9acpd2kbAZ?lm@-75uvH{4R|GInq5gN`c*u!{UbAN&}T9JOt%6XS}x+{gtLsl}s-lI}XAEo(NX1TPh!|a0AE6x0YVI{;o1^3s?$N+vgS^Zq=itW+PQvyREu>_kQqu@H%E1DsA+bEe;Lkx_<r)JA^R52QP3LTMrk6W<1f$A-oo$OOVY;ZB?oIPAg{7N2~{RW`N_kCI|RpT~PO-$AR)Y>Adp2;~T06AF9p?Whx#j2BbDa-!G4`IIY!(wpk2r$P0wpMV=ZCpR!0?lt;M4K314s5LuRQ;Ypi&<UA1?CaDYnZ~ilzThtD5SZ!Rbe6byJaK#pC6~-rzYq8#Yqgu>KWE%he2MmtsQdL$65~9qY!ayj-Ch=12^^O!zi$B^$3Pb&U$dkfH(I%;Qbyq=5_m-A9aklJsl!2gbnx`YJ;@?0j5BZ}tm_o1@;djoCXFZWO_A6aNc=tFWC;pG~W5r&vLn4L2HG(WS&p)+Dgn>(SUw2N}IfKUa7Fo#^#;krpl1FkqWBl)i?;Dho&q(zZWv4*_;wU=!Lmt>F9^~sM$a3Efm}DahB^T5`80t-?xk>k!ZTm(qsO3)@$CTe@&4kPU3Po4h?{5XdLi80e_%pIp5$*%Wmw_}BWY~(*?>fL;9vYA+XBa*i2cP;*rVk1n>?NZM<do48f->YIDIfU-tietkZ5R*7dt0S66;PJXFE4u@YE&JrQgMva{Iq@th-u|%JQKp?B8v@hxAQRGaeJWFDfVVfx|@JoP)(STKf)|=$MQA(1o<K1yd1YagfBH&tHzWl4`(<Yfy34hDb4ZF8q2o2{C0nLNCaOoC1wa;UnP^04kTsZ6e;Pxl9DH+;>X8y@-L$=LTC',
 'postprocess_snapshot.py': 'c-p;MOOND85x)1YNScG~!Q<{7Ew36C(kwfxMJy~DX1UCctSmdLE4N&>Co|pMwXK%8aYP7l;<BUB3ULDo390A6VG#eL`6Wc;!%jb{x>^hLrTxf^j913@Ww?%yk3U^53tlopo1Ci?c3D?K5nAR%Fqt)s)HGYuGhUv>gl!fq&sk1*Nz{swk3S+!T~stDnV=U1DJ#VmRkcozkB^V|MoU^zxhnX=-F_jf@@Ofl4aimag4h?ofLk|6t{TOQqobpoEkO;Mr#LOd8PV*Bz@K2M5v7)vpaL?)`DR3-XDD{5LRMx~4yw<CMlB!yAc!I>^g?jOI;k-^r$xi8aC}1Mq?4$hDu&8Yv4vOoG`ZkvmC9zh<d=F_Vy?+|NRX&a9So)=W=*Y^);n)DwG8#b;M!6)f~8buoX<a^g@k9k%vq`Ck0KQLc^TlSfjxsI1gi_0F{`yPb;%2sVibJ;<@}6E1(w+GByvR`efN8z5r~#~<|kFn$}kX%AR<%}sldS-BM1MMl^~0v%GQ_`V4?^Mx>@9OW=HCHgzud`dY?QX_%DjdA_$^RLZ?yE)Hziw)bb)f)b{;~U7D*fYV#wjL@teqRih5F5OvmJ29+hG9LqzKK`4Nnkibd}qU5Zo7Gdxp!I(h;bz1i|))`RJ4NG*8Sm;6o!CM5MpW0YWn7UO=Cb%WQwr~)w=rA9gGr^bJRBvH4AtqVXluBA{^>c7Vx0QOf>erCTn+@iWu~X0EOA_jr$r)2&Age}XEZxw$h8#@wqMYSmBA2-L$#c=LUQofy-h8MLDHImH1+<M!-d~7h1<RD4@U6Ku^F{ac<({cowW$lXXRa4qLAhk)8I0f?_Ed;UguxT-#%%Skzff}d?sY~syj+og{_ghepU6)>Jo)U|C*=IyS32Qf{>2H}nFNCvnF2czByhIVh-Mk9wMJw<C#RlPIuG<M2yitD<EWn**-)STP~$)dvUvx=EBO+WXb@&M)p#nevStIdNA^9{={53lx$TQsvu!V;ddI;-+;CxUHvYAeQ=4NPHgpXD#Kbu)7n7i-;#I>`DnJ#;Ai?V{b>KBxQ%t^$qC2Ac)F9q;SQoc9Z%T5u;>2mFL#!8ihzlh_S5R>GYx^M6MBT{5Dvb)-Ct6Rx^wpxOmJqML|FH^4n09p?=?5rT-?j4;74QNkn(*R)_8rC({cgr2T&5^5j4)94nN$?22mDJV|B_*hwIXJ9U6H(c<;A%xPexEQm}-h_uT1Xn`!e_##GL5mjdRF)_3m%w8T`9_`^VaPWJa#gw(FoP_pcfbNQnu)HQUJL^!(>l=`b7^$TRi@>}yi6Gn#E<vZSDUjz)m-1um+q0k29BAl6JsvS1KKMyiFPyo3@5w1y!BbHHQGX?ATqCAOWx=vy|m1?x&G@Kwg7OmUFS2hY<GISd?^yo3QIAh{M6L*>uZ-98OIxV`x+stq3eGyLySI_q6{^`<7p?c3ix#&I{Vw{Uawnv>1#+pjb7{eS=Z_diVfDj6kgfVJMHEVKnRuwHe844~9h%Dsi%=9D+l7OAP32U!2aH`navRrT&G;HtMbU#r8+HKh)sr?q+!n0v3_p+8(b1X_e_49YZk<8;b4H?o7Cmv=(X|I$3r-t89WT--GM#l1m&;Os}l)%}s(_}3Gl!Jso6ZeRBBpz}jaJXc}aw;ql=h!_=@UH<x+Gt~IXmX-DG%|8@*)j(nI?T^F!=gNHC>K+DtNCiz>o@Ab<t|-Q6DMjln4&YdQ-wl~`ruhq}h*~_LZ|RN-%>9EvhZ#C7-3G)!9pv2QWT^qreD?-UhIFgp^AFrr?}L+bMCSYYIa=OYdH1gQ@rq^Zy5i+nJ0fXg6hBPAoh;k8V1mQ_h@9#J^)LWCIN8AD!JvDXJ_wPm978YM%k~NOIv}S;bxTLl1tk3Rms+8i{0h&y_WGx6Yp<SfYxnSp0#3DPl)2ZZO%Sr5zdc`Z*wPKu!rPm_XNYgV1!EVxk#MiUME53OQA;;?2?unp4{F-z2Qe`aIQMFkp%#GYtkxU^4(h0z&qo|7xN-<OvvS%EY(Z~E?CF01hW7cjW9;6V1M1-pmcCZrwNnlb=b^r`XH2gp(mNgQVlBoItam^)vA0C6DN}E*ai1xTpu#1~Aj}<E`wZfWEhR~T@SAoL6S~|&PXpW#<6F~p5^M=4gB1+Zj;n-?D-(}N$ske9^u2`5g<}|A7h6ys31Hwxvl<2mMwMVp`K&o^Ut9H4S<+grDwWh*EYx;7Ts?dUBjKXer$;?T#PL-FN~&#Y1Pn>zfT>`TK^>p{x6t-_s=Bqzr(cDvDmKMYm4KHmd*~M|lQBl1XN&Q)j%={YzZUWO-t~N{?@X!%F6Zu3#;qB>eTl~O7_3v(Bwuhh2J~GuFv)KA-8S8;6!+?Y%jq@_G3k|-IhwVDcuZ57`3T7d5VlL2Pj<#adfW)uQXlN+BW|!`l_@?nUTmuA72P+8Hc@s!G^Wgfn18f;TUZqRfJ*jMU*d^%2#k{fH${JdPd)@JgDVB#3!R%RO41amTAE%1X@dvXzETLgrp>`iS`=R8^HAfCQ*MhF1|H031vtkyrODxoO007(G{o{eMI6=_s)FM$RbW1Jf7m&kjjPgaYU}CPNt5H0uxOFyoSv1S8qefGfDPyYYy>*%iYzK<6jPC{_&Ec=p92ojCBO}##2Sjd0xjV&aa}9j0nA$R+3O#)#`?Gvn@js9+Kcl&@G(2<e7fy-ch6xDON*u#xw_uhu)smKVWCg=ct6oQu*o?XqMhg{u;V9#lgtjtp3L?M-eY9$YugMOOg=)J3hG0Fb)U~k2Qp)S-qm$%h}_Y*+u#n{8ppeKpY^eM-pT$pd1gM)Z5lY^g2+l|6}dp3WS9cEfI49)5$;n3FYwGxy0paeJXBTD)GFDNU6wFRCD@>l4Vv^ZIXx)TM$~_&0$rO$R2S&F7cYCKbN5xp>9^}cPJe-s$)<GR)Sl7Z=fJ_?+S>tU+GNkpS3QBxD8QoZ@Mu5|4&9UHd!>|P7J1c0Vb(gZ$M|N@8!bcBCGyDM40<=0;c47C{b4t55_3C<goT<-C`=MK*4l3S`P>_7xIIQn3pORC?M;FD?LvV?gA}M_(MTx5uuh8*frg0TI{=Mt{t!K2$QV$99-^QnJ%nkcJ$PU|?0GVnwL5%#n)BWa4|MITPF06v@OBJof%{01n(qTM^5U*94KEFovE>Jw5Ig!43`!DgC9~N6LuR|<zwtTKZc+EMfppkObN7+`qU#Vm)$$-o5U-C+G5oN@cVa(-OZXZs9dHw>stwPOy*$=-!RL1K*;=|EF`RR%516eQW006&6F|2F>~zhdBOelWv$Y9Ju;VavaMw20RI-DOZ{$Yjk9kW%P6!Y|5TZX{UngJkIvD-p!5^D+gVPR5CXU66h8OMG+x(~_+0Zro%)#h@t_#~$un6JWK>Vvpfs4f)*MZcAPcZP40obtZL*2WK`U|zcez4M`o%DJ3e{%I5+}XIL=K}rX5p+V7G$s5jXB}?1rk828_A(5%<|v0I+xezpHXQ4>Y^<}6{sZF4s&o',
 'priority_probe_snapshot.py': 'c-qZ<O_SS3vf!P+0#6e!0Lmo$L-wqP68%Ec(mX3hBPpb|JsUk82#^GuaDxC1fa-2{^!c6lxVMei*xMfVu%|bNJ<N;uKm1QNGb^)D1r*S1$@AhV9Bu-r%BsrB%F4>Gdi3bghfS2G-gY-nlf`%&UBq6rSj1)NE%MDaDv~nKrrx~BugbXa)^W6qi_%Ln?|uB!E-tHb<mKBc$+IZ+KE!F9CA-b!(W6I$RgrH1Mz+`$MVwWW)vnqVF;v}b^P=)j*9E|Om*?rr>v*xN@?zvgWr&TYaTPCh{VKk$(qu06vR#_;+X25?R5#nWl&6cIldb&z5jrowH&L~g-+8IPZ{eK*`<+pwh~-zg-c?DepLX+Ykt2fin;ZSJN;bM$MFUK-3)R%FNTH<(V6=>-J$UzHo@68Zv5Z$mbRnZo(|CbM)#y6OcQ9ta5tNH|X|+ri)nM@b%l98%y?x`&Jbzmxd685%W9Z#H9)JAGAAI}r`3oojUvCE<{PUl~#BM8p>iLMigLSn@N6~f*6CYt?2iN%Ki|gjwP5Shw+2Q1OqsIr2>DSl($OJfk9c33gpd5t!^Ll*pX7ux#tHhaI<Gbxe0VFhjm6b5-MLZTV!n#LL>TWO?z!<$OuCDUpGK`BNFT%?tTLz0Fx=IlT@%3UjC3wrYijovYSXM=VuP14K6&Jyfz9uU#DU+<MqHGc4=aI)SCIvS;fwwWyZ#nt!;q^;wjbY=zA_k((Jijbc)f4|y0H5Sp2J?9e)RFHh2l$055LT+hCpGdOsI9vNq{SBaD7y)0Ei)QddEwD>;Ahb$hSi#<JA|%=KJxqvD7ah}Nw&(p6;z3r_*VV$s@$V0A7g*$=)y898#shsZE{%aNy!KRa7K5aH~u2cO90K^p)1~1QmtVS*whxr9UC<|5exM~1*vY^012StYF{|2@TbgB_v}~`<7ku7rdGyc4GcTZF5(cmb-*ln<jwQt4Z{{T4J-drR>ai%NE3WHf*&wcE;C80cvA+$JE}-!p&G(En;sn=f9XB;@XxumuoiiKnZ$sKG>?{0{0~uD#)6UnIu}r9yl39mN54A;I$L^kf0-|K8{7$0ZNfJJ)#$)~ST>mZX|A}>M72>DMyjwrZ_{XD05d6%m}v_9OSmZVH~qCLZl+e#RT0N97%Viws=!l#eXF5Z1wX>9&r>Yxd;SDjcz|^$*SMbq{=orOIXFNj8w}5~yb6#&O4#G3({fQH+iFPTY<#J1(s+0dYiEKjZv(s0Ce9Jd+z~|d(ea@!u4v8+zK_yfOe}4rMio;x-{BrqG+CXijniTmn=~oJHUT~m_>0welT^t?R4F##YwqjK05zIUeX9+czh_larSUgE?#jxGyiJaD(LoSYej67+yBnAekM<4N=)8|{k*s)=>Z{9bMO6ijsOA3CE-B&$QCVXj@>(lkSB}edSi*h}RfAPrE!MaN?Er%zEWb>aGl)DdfR9B5@WFM=%+cgqTM%`;t2Mm#w#Y+@k2sodPfy=@3>`KxKqU!J2oXwsO!6JBVs3UK-7r@nto(YpOC4Tg_;-5~3_)xFc_GkWxq<=%?5u*WC~h(Jki}Q9n@Zu4cR5syL+uQN_8Xk<MO;+)H`-Ne{96?;onf4P^qHH0C~{351gR(5U`L@>*pA&BF4pR$s@3j10vTE;nh<kmv@GLPBMkXODrn=;DqMFVm7fGflihX+lM*yA<}AbV*2sJO_}2dpW_bJ@2?QjYnp*B`2r^9!fgkNMj2<Z?*s{wIs&o;BIJ>WiH9eXdg*3DA4QvX?EO3g29%N^zSwzXC!7xWgMOiRcQBs8f6V?&%*IH!L3n%^vSU@p0TBb=BgU}D_a<i3XR6?ULH?oZaF@~#j8Bnb=ALL~Jd_;>YEMX;OReCc+CN-oL%$3%8Sy?sINVEJ3Amdg$+2p`_^DJ5Lq+_q)+a|gWj$!tsPu>^xeh&OP-%NH{f_x}A9C?$khI&|&3|M^_Rm2x@2qW)s=#8;SEs0VaUwD%v@0*z(zA95=n0rmg3jv?7@dKYa$DQT`_{TE8$`0ny;xb>Yh|9;z5_r@mN-|_7EOdDD-SR?Y6oY%$yR_8|O&M_2JX@C3Q~_s=Wq_|o-V@b-!djWe@s{2X%}s$e31OGi0hs(nUMwB6Uq*PmgCpavGI{98V+m-)eTOwPvN*XONx4Nq8O}I)RnBNcBTqFKE|X&BtKX$xv*i!#2$1N}(Uv<xR1fnRp~#m+7a34YvM6a&M(zzDw!nCSNa8sx70r_8@kNq#5s{wL8JGKl2gAt1ehKdrJOIZpVD|*lnF*!sss%D^5g=!0=lB3)gK^W7123@4Lc=(CK1}KQ;U-_=2f`QdW*GQ_cD_{umtG901R<T;36(m41k8{Ql)kH3o`qW<8^HtIHqP+G+YKHt7Q(c{Q6vvhZU~jGf!ko_$QlDY)e)L_6@L!<egJE>#6qN#5Sb#Ze~`7mZpceCHdv==bOSwl0|zDRa|;X877|-Rr~|YR_!&<lEm9f2D!0vHx`5p(%ne%*St1k~JHfvs0CqIqZcoGlYiET`U$qb0@G=HT76|Y5Zs>*?C8doXM6PgJjpC!4sq(iH7ShqM-BG}9a~+zjb=utv2Oa`n&Xb`4K#}IX#Zdu`>j-2V;3o4JdLMhlL460lKPm#gCDNS=Nu#i&1_kf<7rnzG=K8-IbWVvZ@$(5NX<)snovz&lW-VRZ%9;q+PeB*sBH>;dMR|2s%b)59YSJe2pd63K8U;s(e5T|-PDcNbf3!$u=7RZLh`bxk^sXc{F_RyiODt$<0GNheiXvd-z5SpLU!d^BYHsA#lLAGYC1~P>!;y+$CTg&==~st`=Oa&RA3*&PmE{aIrAkj0hN$8R!#!|StKeq$Y2_TWm4IeA?I0kQLtxIfxGA)Skk$ep3tel`q`?vdZ3%7PM9^J44(K{k$9ODhXmS|Fp4!5`k?BU%nB>V>g?0<+6G|e&8=ti~B7Xi{QQM7_`_H?bm3houS1HQs4iW~mLc#S9)%r=+Q9$uOi`o@Q3JvKuJ_$^W$eI2%>_WO)Mx@bb$lbk&5_a5@)H`JO2X05!O7lmg;AXny)jd@DEq>VoUn>P#R1C0o3$n&cPNFhKU3><omgHmn0)<oStlP<O)>X9))ebO|jX>|A3}S`~g<iE|Klg#z(fUiJeIV0MR*jL7;RY>=K>Uq5m6+7jg!^%{s?=jsQ3cOM1lzVlQUO$jsKo=cy(48Fyg1N%?$A5%R*!B)EKOiq@4{Qb?=rc7Qc7QVH@W!vQ46}1jH-MC;`RKdipzl7LA@{lm*z)g#+slt3~*W!%#pW2b~rm6-ZL`-WF|i*U|8GcWpYgi3wT+sqvNlBJ9Bls19p>@sTrp^`SS4Vk@w~C@8}E;gy}ugpPzZZJv<zAtvMTJKgXMPt=_)}7u<speb&mKgEi)I(9PXkON$9VM8cn0YS<HA5~sW*eAG+et{Z~vITNv~XQ6!-z4QWZvIC7E`@LvrjA=FK>&YRU37#At<Nu!E|Naa9?`x$(w=nTqn1N$%Nkan&foWp1Yb}v7kh2=V`LIU-=d*Y%f#3Sbi7hd|%HdJkglA();e-0&U_bTISLzP-B2KMqM|R(5A8O(mC7<KQntcM#s7@0&ZDjt&`y;4=NL4}#Y}%xh^pBI0fP^A85>mb^eT!1GB<17*r(i}pL7_D!GzjP^qD94=QOcRC1myrZ5G(pnP-uyCiSJrUclT!tLiC_tG^x=K#zeAd&h#r&FzH}{I3acU!QPVjN|D$tlU7(;UvQdh`XD__qp<JV2fG9lswq;*%BT6)fZ(Ve_)lvaP2DciqN^(~7k2AlCj8`S?%}Bb7PUVD_orQyLI?5naKxlP^SLz?{RXamdzD5Pp)i0zHDH&tSv!ufp)u;wKd|mCO+Z-T4s_~mT1Q8Fv^*dvF_8_Vc|bfmL>Y2mZpyke+3(s4k?CMBiClCPq0*X!T5IB)^iOL0fotjQH#UC<EOJW=Op~J;<I2BnpL6e~-YvYi^R1EFkYU$M%C%D++c=`mI*+HM`o+U3>oOV46)cB1M1Le(-qV_Ehh#ay;*W&tEuFi1#aj0=#*Xbsle*fap@^%sFhn{G1sf=#%y*y!Miu*gnsYpAgMAjifMk8+6+7jB!M0b>E*)QAqlEBO8$8$X;&KbSD{KKUQ8yl|gtZD?b%lB$GB)C3!k#0DYkH}Q81Kc7B6Vj73im1Gm|Pr?=x-vh(ZKKtdtj1I%5Vi;GwoPv5!Tl60Q7u3LWZ+J!}J`F;XsK=iS{tqzg8g8;qI)_V6`>jHM@z5OLC&X{vCL}U@x?LrpB{Gn{;(UszTLv2+&DHC<F(1v{7>h@6XOkMIL}fR*aw5jyqh=VI=VDsRk(Z!=2z&b$dd?wXJ^9M^YmSDWiQ{QL?IH@epz;!V0wGg5~%hW{m@&3|UTqLl=YWn;Y0bVE0%)MV$l?XBJ=4gedNr7dI1wdGm6i9?3vc7+?^n$*J)GB0`$a1OGA8&3@#<t^{RYKZ+PF9xz>(35bh`Ni^x4oon-}nlxkj>ZbW9T2r?y(oS|UXb5*C&$lO)CRTMwq*xPJ)Aghwa%y8%n<lV5iz0ZNR1z4#9tzh{Z;*nAL?%`TMz-ze!9?=v#ld&)UwfOR1i#P4+N6nQpI9oVJ{S@JvKF&gTds46P0Ws1qPChbPXKgOJ{&4p``4QlJgEt1tvRATxbW!?;54|ZK`u26WSrm(6QNoN%yq(y_MQlUq{tUEv9Q=mSobnHuuRMeAy!KIjvR&@)LlSL&fF(8Ik3Yx+SrY=wLyKFpoe47v@MY7$Q)zsoAyo15MlZdm`7YB3=@g!H0+;02o>w7TpyIl&v6$`HjZs3>9o$vWM#8F{yv|2e&fAUZb=}kgMCy~F=E*R@6P<Gm&1{-xWa*InIMgUxt87%m~u)^2t4n`F^{|!!4XND2?OObGHsql4IMm{7!k8}oaR?_K26dka!o4gKCjuWQVkmoaq4Uqjr4AlBNgCjkp<PEN&<iE3)I6{tC8y8w-~Lnaa5q`5n!k->%jncWlL{5*0r!M)m*cX?Y8PZEILVFlTVWsG<bmdufY$b<M(g_TV=_uK7>0oaE6>h;B;||pg|4H8I)?O!AfdaC=%6^DBR>T@!QW|?o~poX>Y2X1$v+AxxEx8^#7K);a=xdb>s!O;&{3DYG!-Zv;DDoI(MAJngaU0!j*j%Gu-EtaY7DW3`ng{AX|=}X}$(m;1~=qYxLi6uo2z?w)r+7KRYyK4hMf<L&c{kU(B6K>z$Q|W6N*@$H;u;g%wg<OcAbN4@Cbqzct+bl{t?ei+7chyhBvZ&dl@Fpi5oNkv(pQtUfLIpxa7lP?oB{dekJi0~f8eDc4%Ce13VLy{K~$JXjG?pKQ8i4LLcI%bT40^t2?XM&YxOGMV5T7GTg!W6waRK1H2EN2)bTOQMc_N=X7F$X*Mu6<{@3leMPfIVn#LnBY+d#b#AF*?}{`l2ibTN?Ba9JR7f~3O05oJP+NF;rNH@!BT5FM3n}v3|H1kqgFuIh7R&TKps*bK*)}nfxKs1qD2EFKn=sUT5}2@2G77w0F3rRP8}*Vq&|k2U9(On(l@g#Y~8pb8z&sI1RZ8t{S;NRQS#1r02gNs)(T74&3;L@bf;ZMG8(wT3H)bj!2!FvCQn>QGo~t?oj(A{zG%3^Mw9~!TXjnq1j2QE9oW&1076;&CZxh91(QG4U_j109%+By75N1&aZS4KAV{SW^&6adF=TK9x@6#!=?$pjcN<tBC>LzGCJ84(xr6}3%^XFBxTaoQGhqk{(JBnV;X7T8*vk|a8WddaQe32IegQi!u(FwbkVW4-RKe=+Lz!djEFeRFz|fWZ4pMbtt(WtV^P+T$0OJeb2!^zS-JR&(;)Bs;dafbPdWU6$)WNAJSK<rCIzPcA!A_3TLGKRj%8H0%Ty-4G88mt<M7e)SHG7@Kz&YX){r@BqbprOyODI%Nu%eC|nqXu4Xb=G@HJ7{1wgd~O!V)SbP2~C-CGthT=(An<8rM?IKt6{P$`|$24~-E9Rg`zNiSJ$uwco_K1)}+JKTwLNxZu#2;T^7|684rbDi=wjoY>K3pYF=FVHT;U(WWaK&Glrm)4f`&_3pOkR-~8??u^Pp-rg*2eT^othW^1`Ctu9GqxN^f=b8F(-chWH{<9YQHq~ih54wa{3b5VZ15WMIVoU;X_v{M8wof<Wq<u&+SwgD|4_j7b97d0NJrn*uPOU6Wqk><?#Bb1ukSxs)q>9}Krt)Cc^+>oM$)eNF410mX-Q0v_NgX2!87xY(0KR~n0Knb$>}-QF3{Xv)c@Q{R(;cLO>iqox8iK^MTf9&dwm-lYEECK__>Bmq_A36z)}{%&8_AlDU2)c6<{fK9Z#%|Tq-?Lvji`^+wEOu@n_uqH0~J(_vemtoQ?Qt#Qzp7HeKlgg^P`D&DJy4Btf;0zk862wd<Yf4{Ax0kn5Ws%*Z2y8&_b{!%~13kxdWaCUGXAisB&?Xx^~;V3uDUo;K`*BwP<xDT5!vBct|v?wPpeVlJh<`<-3n4*O7LFjD7*5twvS9f$axX+;j>oweZ&z{mMywWI?SRqnH$J$JiO{rm5pa&3T?`l_zEf0fi@0U`0bw{YM(F2k^KNq1NAj4)s?9;NTJIzG{cQ$ViL<1PxzwGe{0j54zL!duOybk9x7j&evsGTc?hie<A1Czo4<RcQAFllp%kEjxSl#QJ9V{>_cY^q>WixFEjuZG4;$l>}&P~6>?w=8_nX2sDU|NIM!j3vI!7{^4BJvN~LF3h*7&UQm*>|g3z!{>5+sQ*6OrBs5b#elpyoq_%6*q<Ogc80((g#e6acq+kOMER?*=eHUZ8DqjZaqW<Zq7b#xKBLWHO*4nlH^2t&ud<#Xx2>hh)r1CoNKGb^yp(SR7Trk;+>^Ua7w6n*|4F^j&9Q)tGDqr?0|*($l{YD)3TE>{XibCTF&s=N+_Goemc#pq6Tz^g=tm33$skbzGWf~N_0V3WVSfpjS3Fmh8)m=d5GWW}g3Qkg%<qxSX?-M6F-QW)dmi|dTI4Or%k+SI=X2xP%Rt9{Gs9mDVlhR3t8uct@1YQLv%Bi%91C7>LY$UWl$t{Ho)PfguOk5k6pt`)W`gLqBIlhU|9(5=EfHz(SqtIw;V?dm9=UzU2fk6RnK)pNnW&Zb=cToOwtRo2W_iUno_G>QUTeP~EEss|U~z_+DT6m>qWfUHlBxo0TGdD@+&z`|{3ngS`sC$l;{K=tVifuFmuhH@&KOMV1bB`R*r;BUOQxZ^EoQ{v8pB_BxQ0>*`EWUf6W2W^&082`ZQV15C2I|^=HC+x<JPTZ6X!m@bPsG&|YgQBk}*6r6y1#tf<s39cVkCqf{_<J=Ml9r0wYeIjr^&loCPFw|8iHn4&CTpP2G<BHhE(=irS014#!Ll^2b{r$!9hg8yj<k6s*~qDc|B{g$6V4cs?3{iN*c*PBKps1AGY)Ec8mYNdxnAg}p5IbibkDo9Ct)?!2-Y)G`32mz%a4P;paTP}K|_RfMvZD*^I6fd^_+F>Pc7DLTiIH^lPw0j7Z}aW&-p+pHea!5YE7E5p=tH~djKDM>R8RXBT3^yl72RhD9IkYH>^{TYRyWvXe;^J&TS5p(}V(POQgm?#3IXrL}!E~BSD``x+vjgWwZiC0_1>$iEx;ubO3p>y4QQIZEi6tJjCL{9k%`V9t8biMzb!jzW?NQGi8`EI{%5bDX|^dm}u(Pq*W=6vm>P3o*Z4BLP(^Jsn#93$f}{S?FOp~I819Q*Cgc833<iZE0hc0vzpC`&`$Z7!c%~T%v=JkN@8^k=zHL}!;PU>1W&aw7}EUI`4u6JLP~kJR!M<*wxY#4WcCg6c_;>J%E*<be*ckPA=Uf+`?o)Qc=<j&efi-u{L|YPFU|8bb_D}Luoo@NvXDGSt394V3&juNJVYF|po|`BjCYJl<neop*IEvy7F1#-r!%8Ct~&PamXBkDdQ9fn%?_Qu!VhDP&d$&@>LfP|pMHbcbs1m#lt%B+?7W^5oM2)pF3Kg(G+p!_YN4|om1!MBaK2n}sOG0wfXSTzONK}00l@~)076}9#G}H`0^@NKV+(wdtEe|Pob4z%F4Q?lsGJTVEtS}W5r-;k6F23ToI+3SX|fmA)hv3$kOQ1L>DaVeoZ%u+pX@a#Rg?kNKlAw$5TWtAH~Hcp;~V~kLAvhDiw){fIovQQnQB4NhkDz{vV(+S7TUz15?HcB`x})<U?La+&%qcD2+F^m)5fXI0NuG%ofm@K>3v}U3MQV@3`jaNYpa`PDVpC`OGV{E6hw_W8*(6!YReO?)#2IUIo7HRYWl)YEjm`DAfzlWEMi8(oGy7bJ~}^>wSGPnZIB`;tq5uedLYMv#2ee&$Nn;dV@H>8P)B@jd!uWFW&WNMcH7~Zt}V27*=zAZ6x+bzDIOEtDHZ^uX*6}F_Z2)e6IYmRNttFKZfBQNCedgP$eqs9D7*AUkZK5*+DbU>V)j||5i>wwW|FI@z<haahs(xD>KdZQs}(E`aOa&kQ4*BqSupU^;&1;uTYJmh%`bobm&&_b|Ly;Rz9aqRU;o#Q_iz90-~Z1qfBl~eFZ<=M|2=^+@MZqD|B*mU^)LVW|Eh_dCRqE3Avzm0NxD!xFoJJn0jplPj>`22$`5)8XBRw5#-UOgQyLTmeM{3c)x35>&u~BoNziypVHp2pKTi_Kds>`AL2n8yULuSQ0G@V&=Gbn@^D)xzQQ&Ok9X7(;1FC0}tDR4lb*hb|uMQVxk~8;8F-twi(kct8ca!hPL2$i;;67c#;w)s5!k3tw?C7h*c9eHx(pdRht4!1^H8duVQKePq^d5O;8pmdm3vyd5<-iDR3Sw4`Tt+%4zk;O$aUFH9;yDvdx!Yo;$u5!VKg9DF^z8@e%3zNiWbT}ZlFU%r^!w5z@;JqC;z;4wu)a)4WFyxPw{x-#FRs+`F|iz&53MT1W=vk?Dgp2=0wFfE@EN6ALWlvlqqi(=G-uA4S*(wB87cHtb|F_kntb94<Wz7wfb`XHZtNR}MHzw6vq`Z-NUaLDECg-Nb?;c+gH&u$B{zjQ@kDa8SQ)5V;oqDzg~!Wj1(e)s?OjrW9t^dWh%qhJx>#<*#bybX4i?k?fJILs06<AeXG$KT7w7zdKa!=jt*Eh?0>nAxt#t_By(CxfWA$EpZph|?MvNT;qgmnbKnSvcBL+xi)oT2;e@|e7Udx{vnD8t5z}u7lD<w<9C=NYhedyN#9vu-G(Jk|>yU8}Ys7K2q!<s|P?Tp*%<`Q&|fqE#}RfCj{VSd^0;sMD=xEOfA8{G(SjT^9s?zPhlOd;+7PX=oCiWsS{8F}iFYZo}VC~|Sq-zP+Sv?Ck(Q*`JN^5Wr?5vXQxi}6-|jCR6Qfv1FB$^nJHdg3~7@0@X)_vVmM6K*Tn(5dG~rA+O!l9*Xy9x1J2=W)bH9P_AUB60fFv$9GsVJqCur=59}8U^Ka=4#`!(u8Ok{69~Q5F>?UcUBJ-65~`#eu{3rPW~--Idto_dJjOI2feP-PRlXr-mrKpt6a18S9{0SL+&|o@8dMn_TtuiI{Cp3jfe_N-3B&E%ET#GUvb!*hml)w#!x!h1F)qP@GQ3&88#X#l0vnQ39(3rBpHV+o0!ZC7HH3k4hoRtLtSjI%mi3X(h}5FyOR<ysc-^c3?necAo#Pwm6(<_BfX@1WO6OjDQCVK2)`at;G~>k!4<9c36)4gxE`obh;yLp*03sgPslq5gR$V@m>!rxPQ9Z$8go7PmPas*zRgEz2u3ZZ(FE#_@u1fRZupW1dT`0gL=!owDV>H!cV09`&*d=^+5W(s!J(1){`uSR#dmK{RBlXtm|{pZCy_?`Ay~QPiBFlc;GcZ^-ShC|+voq)w>9#e8gx|b=zQ_Yc=g9`-@f_hP&YbswyG91h<GRpg-St%_HyJ<mfvo@pkJXgC0ugH#xd`NDuGBt7X!exYz0I}!P3G#1+FujM|kDbYfyDw>ni{}yi}qzvlzAfVWCHO3Pz8-4(}`Sa0h1^+C7>nztKTo^)tpm(ZwYys4?Ns`JHLNGvQe_ixm57Gk)w(E$iSV5}AtV*ZtcgI$8%WD-?_Kl!88N8(HzSx=TvqRW6_-hc$Q|RhQ$V$>F5TCx<Y?fZxHZ@nQAbFDb8U{l;|Fm)i3p0pF1EaT^GKI&Q%-KQuvRW%&dE2^f3?Ry5*o+u}>>BRu0*3&O>GgFQKtQt*7FL0n(XS?6MQR=+)LLHjcvV$i=dwt!yi*EQ@vFC(;<*yYr_qc+@Cat(@bDEma0rC0TE|BN@C6j<a~7ok_jEofi?1p&t@s$Lx%h*vL~2w$L6iCGa}7}e;pS!oO2fRLawDe@Ne*q_EVM(^s~L`>tFE=^mvum)`|5|fcCO#%M8V)NQCNAgN_C*|bi#p~ArgJ>ikQPM1@f^P%h4vARYg2C&Ji>Xo8gK&&>VDbd{uC6gqN(6?~8d9QBXZJF~y9*XzSEMu>bV=p}?zdV1FG4Pg-mREgvGswf_|8nE@LBZxRI1B8@x9_jY|Q?X&DQmppL*Pn1Kq5*1DO^iH<awa#Ta}Ikl@r6Zam0g)4?l-6=Ktcc#M%HE8Zo`fb3?6j$o7BGH+oAU;57E^~B)eYv0`!_SkuAa}B>vPFCcU|An%bg2=v1(RC8rais{U2aF{EJL9a8!WeG&f`yiCRH9na-2y83?Z|ua^7pTvzX|{F{_UI7mv3Io;L?{2edG(}ng|GgeG)!@{d(rZ7C*4{jCk$=hdAB{14v3jXNXdJ1-d*|#j&8iniak30v^a0o&lw-mKc0vlzjK<-Aj58#~Abdhtn5tzdPOkCJ7V^MJt1vF_@8=FUk%&BGGYO-B!3iSitJc!1t(mHAQ>=lzSm>-Ygz1V`L)GFT6(#w1FVFQfeJP^4r}d=RqjsCPMh2goHrfeb`3y5Em8%mY^>W`w=M2v)Ig^ooj&0EB=Nm#{CzAwE~c36BR`|p96%0d7XU%qc5C~rsoZkG|BOdSe~ym0mwMUKmCL{BnaGUFbx*p+H9k-r8pU@lw&B?)^o;{&)rzj%&ol~&WHhGXaL<6<c|dYdFQft=rt;xtEhK%JhYY8V(MNbPYyd$Ab~(bMj*1xgRLhjfH4>TiM*fwrF<Ox<--?4fUa?B(@R%wqq=xa!YW{rq9}CB*3*RiY=a}^e*uR<&`0>)X^@1zL^|N-?%NdF8Ry+T`+ckHO-}o>P-RVONUON4oN|t_6v`a6UKDaIZyTGfW3llgtv2$08;`Sm%x}kdCWLGk8O1d|7O3Z=T7}U(4LQzO=j6Yl9$Kd-k(AVB66Za#mF8d0imn_NJ=_cTZ^$_Vht?j6rK2g!*3mu#l2y&$SEZ_KytYbJ-~7CfE_L`To-y?mVxI-dKB`LmZ;aQ`iMrK}|9xx*C~(j$@2qn$UqGa+1k%>l;TeYZb_dVyqrYgI-^J3dt?<1TdIftzSIsW;rSl)uqv{#k|7SJCi6X<d>I;)Lmz#iVV<?+^04ov0_)wUHd!erU{-TgV2}yXoj6aUc^=7;yx4Hg0>UwafGT&W*Pr$`^n{4l^qioV+%!D^PI{x}_^t;2u+RLb(#+t^vAYs|DnsQur%^Q%=XUovV5aWi%pB(%5g9Z*(0_&gW*LBw{f@*_M??buohVAuMB^xod^8yqE+P*+Lt{=l&zJOf{qNl?m+D7w)%xeBc4zeGJd5CdTg_FCXWj+Qz1oFFF*I3K`Xa0wm;mPxpZ(oM*o}YfJy$X=**^G%xbRx7G@=>r(VCjT-`sz<FL5fh^3xqVEZ+cW@JrjziWJA!>79|;N=VTF>iJN5KE{S7{GI4l$1s7TpJ3zaKBz`btnv?f0pP#-A-@FZfc=hJR+aK;)q%hN@pcI?p1*tmWSZx6hX2yR<GC4E;>+oy&bU&4rPlEV}#u;5iVY_YK&uR_POW5tLg<@T@j#L&FC3J@e6VpZTdoJ0SEIY)w-+hg(S2^fcp_01qvk6WSM_+%q1ZUj!O~hhUukmQ$tcf9*q#b)_N^I>egG2kJC>`21d>(|s=DTkw;m*Tp-;AMByJf>65Y}^=T&%0+4)<6HO)Jg6R~~MnD<@5}TzgqcfF=$d)#m7zt!-eaW~i-SLi0K@$U(*ALEzqi^|ont(&>3cuBqDNV8qwH6QHpk_p<}_`WgDne8YnyiRjq@z4Bdd6zUQi<nRq}{@!ApuA(j?UM9+lhXgd~myyi-BK+1w>J?Wf<Y5WVeX;1Cw)9W$$H0-A3p=HSMxC<eUK_l+!?Jr659w%Y3HEf=QRi3h_g<u;aP|~685q&(xrAGFkZf#nKvGI9l)FP)0hOA#ogRnGO3d6iZqe9Gjq^z|vh4;qJHj`P#5%{Wv^DKKHJHj1))mMpDY(+-Dc<<!iIl2&nYy6zsi%YmmEyBRc^uE>gf`8~IE1O!@F&R*b!wYNjuD-}#!7YA4$2ar{Z_)8?reuf(t@%T$*N=innxo$&2eWSYk#D@VNvdlw3mA*nlbchgZowXh(-wc?4A@%wkpV?QR}HetZkX<UM?!mWw-w!%;f<hn}Zw1_GPwv@rTaDcCm_f+B@%<K@6??*L(6XJm8+|HBfa<`92*QS=TM(_uM{S$0t6>ZNHPE9-s!b7^c(h-*$(0ph4eU*t(yUeKq)U9=d_Gjz=Zm7xX4kHx*H&Ug^9?Rr`QAoY)MWh)1RVD?Z(tOPcC3g-ZxA{XxKKl#W3})R7N;m%%mWtei1gGy+vL(F_?If9i$zcX0}QfZFWgdFvmP^ukfMmG{C?#+W3P`pYeexYIhcna%$PwF<bt'}

EMBEDDED_SHA256 = {'audit_snapshot.py': '75a6118be7c78e17a4fbc1a2c2fd50ddbf4616bdfc5750498a7afda7c808b558',
 'corpus_runner_snapshot.py': '147eed707d4f425d88d0579befaa1c2e43a856a9a2f767ba4785f083249a4c98',
 'crawler_snapshot.py': 'f9b3b3a138efa6104d892847118a11fa7d16387738a32185cf193401184fb761',
 'diagnostics_snapshot.py': 'b544e20569f23328c4df6e6dbc07cacda9f9fb6921bfec77a3fdaec307a2bcaa',
 'export_snapshot.py': 'ac461b827b046ed7ea8fc4488a85d62392e4a95920ca9be7e8d714d3737a4023',
 'postprocess_snapshot.py': '377d747c79f82e52d4fd9f16e36512af540d985536aa0a70b86855eff76842b8',
 'priority_probe_snapshot.py': '5532bc0d02218b11b801981a1b9a07c8a8cfaf60a32ae3c807489ece3907e384'}


RUN_DIR.mkdir(parents=True, exist_ok=True)
POSTPROCESS_FILES = {'audit_snapshot.py', 'export_snapshot.py', 'diagnostics_snapshot.py', 'postprocess_snapshot.py'}
for name, encoded in EMBEDDED_FILES.items():
    target = RUN_DIR / name
    if name in POSTPROCESS_FILES or (not target.exists() and not POSTPROCESS_ONLY):
        body = zlib.decompress(base64.b85decode(encoded))
        if hashlib.sha256(body).hexdigest() != EMBEDDED_SHA256[name]:
            raise ValueError(f'Snapshot bị hỏng: {name}')
        target.write_bytes(body)
CRAWLER = RUN_DIR / 'crawler_snapshot.py'
CORPUS_RUNNER = RUN_DIR / 'corpus_runner_snapshot.py'
AUDITOR = RUN_DIR / 'audit_snapshot.py'
EXPORTER = RUN_DIR / 'export_snapshot.py'
DIAGNOSTICS = RUN_DIR / 'diagnostics_snapshot.py'
import importlib.util
postprocess_spec = importlib.util.spec_from_file_location('notebook_postprocess', RUN_DIR / 'postprocess_snapshot.py')
POSTPROCESS = importlib.util.module_from_spec(postprocess_spec)
postprocess_spec.loader.exec_module(POSTPROCESS)
diagnostics_archive = WORKDIR / f'{RUN_DIR.name}_diagnostics.zip'
from contextlib import contextmanager
import traceback
LOG_DIR = RUN_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def refresh_diagnostics(strict=False):
    try:
        result = subprocess.run(
            [sys.executable, '-u', str(DIAGNOSTICS), '--out', str(RUN_DIR), '--archive', str(diagnostics_archive)],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace',
        )
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(result.stdout)
    except (OSError, subprocess.CalledProcessError) as exc:
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(str(exc) + '\n' + str(getattr(exc, 'output', '') or '') + '\n')
        if strict:
            raise
        print(f'Tạo ZIP chẩn đoán thất bại: {exc}', flush=True)

def save_checkpoint_archive(failed_step=None, error=None):
    archive = POSTPROCESS.package_run(
        RUN_DIR, WORKDIR, id_from=ID_FROM, id_to=ID_TO, dataset_revision=DATASET_REVISION,
        source_sha256=SOURCE_SHA256, packages=versions, refresh_diagnostics=refresh_diagnostics,
        failed_step=failed_step, error=error)
    print('ZIP checkpoint đầy đủ:', archive, flush=True)
    print('ZIP chẩn đoán:', diagnostics_archive, flush=True)
    return archive

@contextmanager
def diagnostic_step(name):
    try:
        yield
    except BaseException as exc:
        with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
            stream.write(f'Bước: {name}\n' + traceback.format_exc() + '\n')
        try:
            save_checkpoint_archive(failed_step=name, error=f'{type(exc).__name__}: {exc}')
        except Exception as rescue_error:
            print(f'Đóng gói cứu dữ liệu lỗi: {rescue_error}; checkpoint vẫn ở {RUN_DIR}', flush=True)
            with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
                stream.write('Đóng gói cứu dữ liệu: ' + traceback.format_exc() + '\n')
        raise
    else:
        refresh_diagnostics()
with diagnostic_step('snapshot_validation'):
    if (RUN_DIR / 'crawl_config.json').exists():
        saved_config = json.loads((RUN_DIR / 'crawl_config.json').read_text(encoding='utf-8'))
        if hashlib.sha256(CRAWLER.read_bytes()).hexdigest() != saved_config['script_sha256']:
            raise ValueError('Snapshot không khớp hash của run cũ; không được ép resume.')
    if not requirements_path.exists():
        requirements_path.write_text(''.join(f'{k}=={v}\n' for k, v in versions.items()), encoding='utf-8')
print('Đã chuẩn bị snapshot và ghi phiên bản dependencies.')
refresh_diagnostics()


## 4. Tải hoặc đọc danh sách URL

Nguồn được cố định theo revision và kiểm tra SHA-256. Chỉ đọc corpus URL;
không dùng query làm nội dung hay nhãn. File Input có thể ở vị trí khác nhưng
được copy vào đường dẫn ổn định trong `/kaggle/working` để dùng lại khi resume.


In [ ]:
with diagnostic_step('source'):
    import requests

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if SOURCE_PARQUET:
        supplied = Path(SOURCE_PARQUET)
        if not supplied.is_file():
            raise FileNotFoundError(supplied)
        if supplied.resolve() != SOURCE_PATH.resolve():
            shutil.copyfile(supplied, SOURCE_PATH)
    elif not SOURCE_PATH.exists():
        temporary = SOURCE_PATH.with_suffix('.download')
        try:
            with requests.get(SOURCE_URL, stream=True, timeout=(15, 90)) as response:
                response.raise_for_status()
                with temporary.open('wb') as stream:
                    for block in response.iter_content(1024 * 1024):
                        if block:
                            stream.write(block)
            if sha256_file(temporary) != SOURCE_SHA256:
                raise ValueError('Hash file tải về không khớp nguồn đã chốt.')
            temporary.replace(SOURCE_PATH)
        finally:
            temporary.unlink(missing_ok=True)
    if sha256_file(SOURCE_PATH) != SOURCE_SHA256:
        raise ValueError('Parquet không khớp nguồn đã chốt. Kiểm tra file Input.')
    print(f'Nguồn hợp lệ: {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} byte)')


## 5. Chọn ID và tạo manifest

Nếu khoảng chọn có ID không tồn tại trong nguồn, notebook báo số lượng thiếu,
rồi lấy đúng các hàng thực sự có trong khoảng; không tạo ID hoặc URL thay thế.
Bước chuẩn bị hiện quét toàn file để kiểm tra nguồn và thống kê hostname, nên có thể mất một lúc.


In [ ]:
with diagnostic_step('prepare'):
    import pyarrow.parquet as pq
    from collections import Counter
    from urllib.parse import urlsplit
    from IPython.display import display
    from datetime import datetime, timezone
    import time
    import uuid

    selected = pq.read_table(SOURCE_PATH, filters=[('id', '>=', ID_FROM), ('id', '<=', ID_TO)]).to_pylist()
    selected.sort(key=lambda row: row['id'])
    if not selected:
        raise ValueError('Không có URL nào trong khoảng ID đã chọn.')
    if len({row['id'] for row in selected}) != len(selected):
        raise ValueError('Nguồn có ID trùng trong khoảng chọn.')
    missing_count = ID_TO - ID_FROM + 1 - len(selected)
    print(f'Chọn {len(selected):,} URL; có {missing_count:,} ID không tồn tại trong khoảng.')
    print('Phân bố hostname:', dict(Counter(urlsplit(row['url']).hostname for row in selected)))
    display(selected[:5])

    LOG_DIR = RUN_DIR / 'logs'
    LOG_DIR.mkdir(exist_ok=True)

    def log_event(name, event):
        with (LOG_DIR / name).open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')

    def run_script(script, *arguments):
        # Output hiển thị trực tiếp để theo dõi tiến độ; không dùng shell.
        invocation_id = uuid.uuid4().hex
        started = time.monotonic()
        operation = str(arguments[0]) if script in (CRAWLER, CORPUS_RUNNER) else Path(script).stem
        log_event('invocations.jsonl', {
            'event': 'start', 'invocation_id': invocation_id, 'operation': operation,
            'at': datetime.now(timezone.utc).isoformat(), 'arguments': list(map(str, arguments)),
        })
        process = subprocess.Popen(
            [sys.executable, '-u', str(script), *map(str, arguments)],
            cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        returncode = None
        try:
            with (LOG_DIR / (Path(script).stem + '.log')).open('a', encoding='utf-8') as log:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    if operation == 'crawl':
                        try:
                            progress = json.loads(line)
                        except ValueError:
                            progress = {}
                        if isinstance(progress, dict) and 'completed_this_invocation' in progress:
                            log_event('progress.jsonl', {
                                **progress, 'invocation_id': invocation_id,
                                'at': datetime.now(timezone.utc).isoformat(),
                                'elapsed_seconds': time.monotonic() - started,
                            })
                            refresh_diagnostics()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
            log_event('invocations.jsonl', {
                'event': 'end', 'invocation_id': invocation_id, 'operation': operation,
                'at': datetime.now(timezone.utc).isoformat(),
                'elapsed_seconds': time.monotonic() - started,
                'returncode': process.returncode,
            })
            refresh_diagnostics()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)

    manifest_path = RUN_DIR / 'manifest.json'
    if not manifest_path.exists():
        run_script(CRAWLER, 'prepare', '--source', SOURCE_PATH, '--out', RUN_DIR,
                   '--size', len(selected), '--id-start', ID_FROM, '--id-stop', ID_TO + 1)
    else:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['source_sha256'] != SOURCE_SHA256 or manifest['selected'] != len(selected):
            raise ValueError('Manifest không khớp nguồn hoặc số URL đã chọn.')
        if not POSTPROCESS_ONLY and Path(manifest['source']).resolve() != SOURCE_PATH.resolve():
            raise ValueError('Đường dẫn nguồn của run cũ khác notebook này; dùng run mới.')
        sample_bytes = (RUN_DIR / 'sample.jsonl').read_bytes()
        if hashlib.sha256(sample_bytes).hexdigest() != manifest['sample_sha256']:
            raise ValueError('Hash sample của run cũ đã thay đổi.')
        with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
            sample = [json.loads(line) for line in stream if line.strip()]
        if {r['id']: r['url'] for r in sample} != {r['id']: r['url'] for r in selected} or len(sample) != len(selected):
            raise ValueError('ID/URL của run cũ không khớp khoảng chọn.')
        print('Manifest hợp lệ; giữ checkpoint để resume.')


## 6. Crawl — chạy lại ô này để tiếp tục

Tiến độ in mỗi 25 URL hoàn tất hoặc khi hết lượt. Chế độ robots theo `RESPECT_ROBOTS`,
giới hạn một request cùng registered domain, có retry và checkpoint theo ID.
`WORKERS = 4` không có nghĩa bốn request đồng thời đến cùng website.
ID đã có checkpoint, kể cả thất bại, sẽ được bỏ qua khi resume.
`MAX_NEW` giới hạn URL Requests mới; Selenium tiếp tục các URL lỗi, text rỗng hoặc
QC lỗi đã có checkpoint nhưng chưa thử browser. Chrome được kiểm tra trước lượt Requests;
nếu không khởi động được, ô crawl dừng và ghi lỗi riêng, giữ checkpoint hiện có.
Kết quả cuối ưu tiên engine có QC tốt hơn, giữ Requests khi hai kết quả tương đương.
Parser 120ask giữ câu hỏi/câu trả lời; cnkang giữ thân bài, bỏ menu và bài liên quan.
`records/` là kết quả cuối dùng để audit/export; `requests_records/` và
`browser_records/` giữ bằng chứng của từng engine. Chỉ nội dung qua QC mới được xuất corpus.
ZIP `<run>_diagnostics.zip` được cập nhật sau mỗi log tiến độ và khi lệnh kết thúc,
kể cả ngắt ô crawl. File có một số mẫu record/raw để debug, không nén lại toàn bộ raw.

Mặc định `BROWSER_TEST_MODE = 'failed_only'`: sau khi Requests hết lượt retry,
mọi lỗi HTTP/mạng/timeout/challenge, lỗi trích xuất, text rỗng và QC `review`/`rejected`
đều tự được đưa sang Selenium, kể cả PDF cần kiểm tra lại. Log `selenium_fallback_scheduled`
ghi số URL được chuyển và lý do. `records/` giữ `selenium_fallback_reason` cùng lỗi Requests gốc;
nội dung Selenium có QC tốt hơn được chọn và xuất vào corpus nếu đạt `accepted`.
Chrome/driver không khởi động được sẽ dừng sớm và lưu lỗi riêng. Selenium vẫn có thể thất bại.
`off` là lựa chọn chủ động tắt fallback; robots bị chặn khi `RESPECT_ROBOTS = True`
và URL không hợp lệ/địa chỉ không được phép vẫn không được chuyển sang browser.
Sau cập nhật snapshot, dùng `RUN_TAG` mới; có thể đặt `RECOVERY_RUN_PATH` để phục hồi run cũ.

Tiến độ và cảnh báo QC (review, domain_parser_fallback, …) được tổng hợp mỗi LOG_EVERY URL.
Lỗi HTTP/mạng, trang challenge và nội dung rỗng vẫn in ngay kèm URL và nguyên nhân.


In [ ]:
with diagnostic_step('crawl'):
    if POSTPROCESS_ONLY:
        print('POSTPROCESS_ONLY: bỏ qua Requests/Selenium; giữ nguyên cấu hình và snapshot crawl.')
    else:
        POSTPROCESS.invalidate(RUN_DIR)
        corpus_settings = {
            'user_agent': USER_AGENT, 'delay': DELAY, 'attempts': ATTEMPTS,
            'timeout': [PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT], 'budget': PAGE_REQUEST_BUDGET,
            'respect_robots': RESPECT_ROBOTS,
            'robots_options': {'robots_attempts': ROBOTS_ATTEMPTS,
                               'robots_connect_timeout': ROBOTS_CONNECT_TIMEOUT,
                               'robots_read_timeout': ROBOTS_READ_TIMEOUT,
                               'robots_retry_budget': ROBOTS_RETRY_BUDGET,
                               'robots_failure_ttl': ROBOTS_FAILURE_TTL},
            'browser_mode': BROWSER_TEST_MODE, 'browser_user_agent': BROWSER_USER_AGENT,
            'chrome_binary': str(CHROME_BINARY) if CHROME_BINARY else None,
            'browser_load_timeout': BROWSER_LOAD_TIMEOUT, 'browser_wait_seconds': BROWSER_WAIT_SECONDS,
            'browser_attempts': BROWSER_ATTEMPTS,
        }
        settings_path = RUN_DIR / 'corpus_settings.json'
        settings_path.write_text(json.dumps(corpus_settings, ensure_ascii=False, indent=2), encoding='utf-8')
        arguments = ['crawl', '--out', RUN_DIR, '--workers', WORKERS, '--config', settings_path]
        if '--log-every' in CORPUS_RUNNER.read_text(encoding='utf-8'):
            arguments += ['--log-every', LOG_EVERY]
        if RECOVERY_RUN_PATH:
            arguments += ['--recovery-source', RECOVERY_RUN_PATH]
        if MAX_NEW is not None:
            arguments += ['--max-new', MAX_NEW]
        print('ZIP chẩn đoán tự cập nhật:', diagnostics_archive)
        run_script(CORPUS_RUNNER, *arguments)


## 7. Báo cáo, kiểm tra và xuất corpus

Audit chỉ chạy khi đủ checkpoint; raw rỗng hoặc HTML không parse được được ghi cờ chẩn đoán.
Sai hash, ID/URL hoặc thiếu raw của bản ghi extracted vẫn làm bước này thất bại.
Corpus chỉ xuất sau audit đạt. Thành công được ghi riêng trong postprocess_status.json;
run hết pending chưa đủ điều kiện thành công/upload. Nếu lỗi, notebook tạo ZIP checkpoint
đầy đủ và diagnostics trước khi báo lại lỗi gốc. Accepted là QC tự động, vẫn cần review nội dung.


In [ ]:
with diagnostic_step('report_audit_export'):
    summary = POSTPROCESS.run_postprocess(RUN_DIR, SOURCE_PATH, run_script)
    display({k: summary[k] for k in ['selected', 'completed', 'pending', 'accepted', 'review', 'rejected', 'status_counts']})
    if (RUN_DIR / 'access_summary.json').exists():
        display(json.loads((RUN_DIR / 'access_summary.json').read_text(encoding='utf-8')))
        print('Lỗi theo tổng URL dataset giảm dần:', RUN_DIR / 'WEB_ERRORS.md')
        print('Đối chiếu Requests/Selenium:', RUN_DIR / 'requests_vs_selenium.csv')
    if summary['pending']:
        print('Run chưa hoàn tất; chưa audit/export hoặc đánh dấu thành công. Chạy tiếp ô crawl.')
    elif summary['accepted'] > 0:
        corpus = pq.read_table(RUN_DIR / 'corpus.parquet')
        print(corpus.schema)
        display(corpus.select(['id', 'url', 'title']).slice(0, 5).to_pylist())
    else:
        print('Audit đạt; không có tài liệu accepted để xuất corpus. Xem QUALITY_REPORT.md.')


## 8. ZIP chẩn đoán — tích hợp trong Output

Chạy ô này để tạo lại gói debug từ checkpoint hiện có, kể cả khi crawl chưa xong.
Không cần file Python riêng, không tải lại URL. ZIP chứa:
- `overview.html`, `summary.json`: tiến độ, tốc độ/ETA, trạng thái HTTP và QC.
- CSV từng ID, lỗi/pending, request/retry và redirect.
- Log pip/crawl/Chrome, kết quả preflight, traceback notebook, manifest/cấu hình và phiên bản môi trường.
- Snapshot code, bằng chứng robots và tối đa 10 mẫu record/raw để kiểm tra parser.

Raw mẫu có ngân sách 10 MiB; các mẫu bị bỏ qua được ghi trong `sample_manifest.json`.
Giải nén ZIP và mở `overview.html`. Có thể gửi nguyên ZIP này để chẩn đoán lỗi.


In [ ]:
with diagnostic_step('diagnostics_archive'):
    from IPython.display import FileLink
    import zipfile

    refresh_diagnostics(strict=True)
    with zipfile.ZipFile(diagnostics_archive) as diagnostic_bundle:
        diagnostic_summary = json.loads(diagnostic_bundle.read('summary.json'))
    display({key: diagnostic_summary[key] for key in ['selected', 'completed', 'pending', 'quality_status_counts', 'timing']})
    print('ZIP chẩn đoán trong Output:', diagnostics_archive)
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))


## 9. Đóng gói corpus đầy đủ để tải về

ZIP được tạo cả khi xử lý thất bại; manifest ghi complete = False cho run chưa qua audit/export.
ZIP chứa cả raw, records, snapshot code và corpus để truy vết/resume; có thể khá lớn.
ZIP riêng `<run>_diagnostics.zip` nhỏ hơn: giải nén và mở `overview.html` để xem
tiến độ, tốc độ/ETA, HTTP status, lỗi và cờ chất lượng. Có CSV từng ID và log cài dependencies.
Tải file trong Output hoặc qua link bên dưới. Khi đổi session, cần giữ output hoặc
thêm run đã giải nén làm Kaggle Input và đặt `RESUME_RUN_PATH` trước khi chạy lại.
Resume không tự thử lại ID lỗi đã có checkpoint. Dùng `RECOVERY_RUN_PATH`
cùng `RUN_TAG` mới để thử lại lỗi bằng code/cấu hình mới và tái sử dụng raw thành công.


In [ ]:
with diagnostic_step('checkpoint_archive'):
    from IPython.display import FileLink

    archive = save_checkpoint_archive()
    notebook_manifest = json.loads((RUN_DIR / 'notebook_manifest.json').read_text(encoding='utf-8'))
    print('Run đã qua audit/export:', notebook_manifest['complete'])
    display(FileLink(str(Path(archive).relative_to(WORKDIR))))
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))
    if notebook_manifest['complete'] and notebook_manifest['accepted'] > 0:
        display(FileLink(str((RUN_DIR / 'corpus.parquet').relative_to(WORKDIR))))
        display(FileLink(str((RUN_DIR / 'corpus.jsonl').relative_to(WORKDIR))))


## 10. Tự upload output lên Hugging Face Dataset

Repo đích: **AncientWings/AncientWings_R2AI_STAGE3_ds**. Bật Internet và gắn Kaggle Secret
**HF_TOKEN** có quyền ghi vào repo. Run All sẽ tự chạy ô này sau khi xuất corpus/đóng ZIP.
Không cần đăng nhập qua widget hoặc tải output về máy.

Mỗi run được phân biệt bằng `RUN_DIR.name`: mặc định `ids_<ID_FROM>_<ID_TO>_synced`.
Recovery có tên riêng; nếu crawl lại cùng khoảng ID với code/cấu hình khác, đặt **RUN_TAG mới**
ở mục 1. Chạy lại ô upload cùng run sẽ cập nhật đúng các đường dẫn của run đó.

```text
corpus/<run>.parquet                       # Đầu vào RAG: chỉ tài liệu accepted
corpus_jsonl/<run>.jsonl                   # Cùng corpus, định dạng JSONL
archives/<run>.zip                         # Chỉ khi bật HF_UPLOAD_ARCHIVE = True
diagnostics/<run>_diagnostics.zip           # Gói chẩn đoán
manifests/<run>/notebook_manifest.json      # Khoảng ID và số lượng bản ghi
manifests/<run>/upload_manifest.json        # Danh sách file, kích thước và SHA-256
```

Mặc định upload corpus, diagnostics và manifest; không upload ZIP raw/checkpoint (`HF_UPLOAD_ARCHIVE = False`).
Mục 9 vẫn tạo ZIP đầy đủ trong Kaggle Output để tải về/resume; chỉ bật upload archive khi chủ động cần.
Đặt `HF_UPLOAD_ENABLED = False` nếu chỉ muốn giữ output Kaggle.
Run còn pending (ví dụ thử `MAX_NEW = 10`) sẽ **chưa upload**; chạy tiếp mục 6 rồi chạy lại mục 7–10.
Run hoàn tất nhưng không có accepted vẫn upload manifest/ZIP; không xuất corpus cũ còn sót lại.
Upload đọc file từ đĩa, không nạp cả ZIP vào RAM. Nếu mạng lỗi, ô thử lại tối đa 3 lần;
có thể chạy lại riêng mục 10 mà không crawl lại. Kết quả commit được lưu tại
`hf_upload_result.json` trong run. Repo đích phải tồn tại; token/repo lỗi sẽ được báo ở ô này.

Tài liệu: [Hugging Face upload](https://huggingface.co/docs/huggingface_hub/en/guides/upload)
và [Kaggle Secrets](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md#using-secrets-in-kernels).


In [ ]:
# ===== UPLOAD OUTPUT CỦA RUN NÀY LÊN HUGGING FACE =====
HF_UPLOAD_ENABLED = True
HF_REPO_ID = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
HF_SECRET_NAME = 'HF_TOKEN'  # Tạo trong Kaggle: Add-ons > Secrets; không dán token vào code.
HF_UPLOAD_JSONL = True
HF_UPLOAD_ARCHIVE = False  # ZIP raw/checkpoint chỉ giữ trong Kaggle Output.
HF_UPLOAD_DIAGNOSTICS = True
HF_UPLOAD_ATTEMPTS = 3


def upload_current_run_to_hf():
    import hashlib
    import importlib.metadata
    import json
    from datetime import datetime, timezone
    from pathlib import Path
    import re
    import subprocess
    import sys
    import time
    import traceback
    from urllib.parse import quote

    if not HF_UPLOAD_ENABLED:
        print('Đã tắt upload Hugging Face; output vẫn được lưu trong Kaggle Output.')
        return {'status': 'disabled'}

    run_dir = Path(RUN_DIR)
    manifest_path = run_dir / 'notebook_manifest.json'
    if not manifest_path.is_file():
        raise RuntimeError('Chạy mục 7–9 để xuất corpus và đóng ZIP trước khi upload.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (manifest.get('id_from_inclusive'), manifest.get('id_to_inclusive')) != (ID_FROM, ID_TO):
        raise RuntimeError('Manifest không khớp khoảng ID hiện tại; chạy lại mục 7–9.')
    if manifest.get('dataset_revision') != DATASET_REVISION:
        raise RuntimeError('Manifest không khớp revision nguồn; chạy lại mục 7–9.')
    if manifest.get('pending', 1) != 0:
        print(f"Chưa upload: run còn {manifest.get('pending', '?')} URL pending. "
              'Chạy tiếp mục 6, rồi chạy lại mục 7–10.')
        return {'status': 'incomplete', 'run_name': run_dir.name}
    # Completion requires fresh audit/export evidence, even if a legacy manifest says complete.
    import importlib.util
    helper_path = run_dir / 'postprocess_snapshot.py'
    if not helper_path.is_file():
        raise RuntimeError('Thiếu bằng chứng audit/export mới; chạy lại mục 7–9 với POSTPROCESS_ONLY = True.')
    helper_spec = importlib.util.spec_from_file_location('hf_postprocess_gate', helper_path)
    helper = importlib.util.module_from_spec(helper_spec)
    helper_spec.loader.exec_module(helper)
    helper.require_ready(run_dir, manifest)
    if type(HF_UPLOAD_ATTEMPTS) is not int or HF_UPLOAD_ATTEMPTS < 1:
        raise ValueError('HF_UPLOAD_ATTEMPTS phải là số nguyên dương.')

    run_name = run_dir.name
    files = [(manifest_path, f'manifests/{run_name}/notebook_manifest.json')]
    if manifest.get('accepted', 0) > 0:
        files.append((run_dir / 'corpus.parquet', f'corpus/{run_name}.parquet'))
        if HF_UPLOAD_JSONL:
            files.append((run_dir / 'corpus.jsonl', f'corpus_jsonl/{run_name}.jsonl'))
    else:
        print('Run không có tài liệu accepted; chỉ upload manifest và các ZIP đã chọn.')
    if HF_UPLOAD_ARCHIVE:
        files.append((Path(WORKDIR) / f'{run_name}.zip', f'archives/{run_name}.zip'))
    if HF_UPLOAD_DIAGNOSTICS:
        files.append((Path(WORKDIR) / f'{run_name}_diagnostics.zip',
                      f'diagnostics/{run_name}_diagnostics.zip'))
    for local_path, _ in files:
        if not local_path.is_file() or local_path.stat().st_size == 0:
            raise RuntimeError(f'Thiếu output hoặc file rỗng: {local_path.name}. Chạy lại mục 7–9.')

    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(HF_SECRET_NAME)
    except Exception:
        raise RuntimeError(f'Tạo và bật Kaggle Secret {HF_SECRET_NAME!r} cho notebook này; '
                           'token cần quyền ghi vào dataset đích.') from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError(f'Kaggle Secret {HF_SECRET_NAME!r} đang rỗng.')
    token = token.strip()

    try:
        from huggingface_hub import HfApi, CommitOperationAdd
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        'huggingface_hub>=0.34,<3'], check=True)
        from huggingface_hub import HfApi, CommitOperationAdd

    def report_upload_error(error, stage, attempt=None):
        # Giữ thông tin chẩn đoán, nhưng che token/header xác thực và query của URL ký.
        def redact(value):
            value = str(value).replace(token, '[REDACTED_TOKEN]')
            value = re.sub(r'\bhf_[A-Za-z0-9]{8,}\b', '[REDACTED_TOKEN]', value)
            value = re.sub(r'(?i)\bBearer\s+[A-Za-z0-9._~+/=-]+',
                           'Bearer [REDACTED_TOKEN]', value)
            value = re.sub(r'(https?://)[^/\s@]+@', r'\1[REDACTED_AUTH]@', value)
            value = re.sub(r'''(https?://[^\s?<>"'\)\]]+)\?[^\s<>"'\)\]]+''',
                           r'\1?[REDACTED_QUERY]', value)
            value = re.sub(r'''(?i)(\bauthorization\b["']?\s*[:=]\s*["']?)'''
                           r'''(?:(?:Bearer|Basic)\s+)?[^\s"',;}]+''',
                           r'\1[REDACTED_AUTH]', value)
            value = re.sub(r'''(?i)(\b(?:access[_-]?token|token|api[_-]?key)\b'''
                           r'''["']?\s*[:=]\s*["']?)[^\s"',;}&]+''',
                           r'\1[REDACTED_TOKEN]', value)
            return value

        chain, seen, current, status = [], set(), error, None
        while current is not None and id(current) not in seen and len(chain) < 8:
            seen.add(id(current))
            message = redact(str(current))
            chain.append(f'{type(current).__name__}: {message}')
            code = getattr(getattr(current, 'response', None), 'status_code', None)
            if status is None and isinstance(code, int) and 100 <= code <= 599:
                status = code
            if status is None:
                # hf_xet có thể bọc HTTP error trong RuntimeError mà không có response.
                match = re.search(r'\b(?:HTTP(?:\s+(?:status|error|code|client|server)){0,4}'
                                  r'|status(?:\s+code)?)\s*[:=]?\s*\(?([1-5]\d{2})\b',
                                  message, re.IGNORECASE)
                if match:
                    status = int(match.group(1))
            current = current.__cause__ if current.__cause__ is not None else current.__context__
        versions = {}
        for package in ('huggingface-hub', 'hf-xet'):
            try:
                versions[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                versions[package] = 'not installed / unavailable'
        detail = '\nCaused by: '.join(chain)
        heading = (f'Hugging Face error | stage={stage} | '
                   f'attempt={attempt or "-"}/{HF_UPLOAD_ATTEMPTS} | HTTP {status or "?"}')
        print(heading)
        print(detail)
        print('SDK:', json.dumps(versions, ensure_ascii=False))
        diagnostic = {
            'time_utc': datetime.now(timezone.utc).isoformat(), 'stage': stage,
            'attempt': attempt, 'http_status': status, 'repo_id': HF_REPO_ID,
            'run_name': run_name, 'sdk_versions': versions, 'error_chain': chain,
        }
        log_path = run_dir / 'logs' / 'hf_upload_error.log'
        try:
            log_path.parent.mkdir(parents=True, exist_ok=True)
            with log_path.open('a', encoding='utf-8') as log:
                log.write(redact(json.dumps(diagnostic, ensure_ascii=False)) + '\n')
                log.write(redact(''.join(traceback.format_exception(
                    type(error), error, error.__traceback__))) + '\n')
            print('Log lỗi upload:', log_path)
        except OSError:
            print('Không ghi được file log; thông báo lỗi đã in ở trên.')
        return status, detail

    api = HfApi(token=token)  # Không login(), không lưu token vào file hay biến môi trường.
    try:
        api.repo_info(repo_id=HF_REPO_ID, repo_type='dataset')
    except Exception as error:
        status, detail = report_upload_error(error, 'repo_info')
        raise RuntimeError(f'Không truy cập được dataset {HF_REPO_ID} (HTTP {status or "?"}). '
                           f'\nLỗi gốc: {detail}\n'
                           'Kiểm tra Internet, repo đích và quyền của HF_TOKEN.') from None

    file_records = []
    for local_path, remote_path in files:
        digest = hashlib.sha256()
        with local_path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        file_records.append({'path_in_repo': remote_path, 'bytes': local_path.stat().st_size,
                             'sha256': digest.hexdigest()})
    upload_manifest = {
        'format_version': 1,
        'source_notebook': 'crawl_corpus_kaggle_hf_upload.ipynb',
        'repo_id': HF_REPO_ID, 'run_name': run_name,
        'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
        'dataset_revision': manifest['dataset_revision'],
        'source_sha256': manifest['source_sha256'],
        'complete': True, 'corpus_policy': 'accepted_only',
        'counts': {key: manifest[key] for key in ('selected', 'completed', 'pending', 'accepted')},
        'prepared_at_utc': datetime.now(timezone.utc).isoformat(),
        'files': file_records,
    }
    upload_manifest_path = run_dir / 'hf_upload_manifest.json'
    upload_manifest_path.write_text(json.dumps(upload_manifest, ensure_ascii=False, indent=2) + '\n',
                                    encoding='utf-8')
    files.append((upload_manifest_path, f'manifests/{run_name}/upload_manifest.json'))
    print(f'Dataset đích: {HF_REPO_ID} | Run: {run_name} | Accepted: {manifest["accepted"]}')
    for local_path, remote_path in files:
        print(f'  {remote_path} ({local_path.stat().st_size / 1024**2:.2f} MiB)')

    for attempt in range(1, HF_UPLOAD_ATTEMPTS + 1):
        try:
            # Một commit cho corpus, manifest và ZIP của cùng run; chỉ thêm/cập nhật các path này.
            # Tạo lại operations khi retry vì SDK có thể thay đổi chúng trong quá trình upload.
            commit = api.create_commit(
                repo_id=HF_REPO_ID, repo_type='dataset',
                operations=[CommitOperationAdd(path_in_repo=remote, path_or_fileobj=str(local))
                            for local, remote in files],
                commit_message=f'Upload crawl {run_name} ({manifest["accepted"]} accepted)',
            )
            break
        except Exception as error:
            status, detail = report_upload_error(error, 'create_commit', attempt)
            retryable = status is None or status >= 500 or status in (408, 409, 429)
            if not retryable or attempt == HF_UPLOAD_ATTEMPTS:
                raise RuntimeError(f'Upload chưa thành công (HTTP {status or "?"}, '
                                   f'{type(error).__name__}).\nLỗi gốc: {detail}\n'
                                   'Xem logs/hf_upload_error.log để biết chi tiết; '
                                   'output local vẫn còn, có thể chạy lại riêng mục 10.') from None
            print(f'Upload thất bại (lượt {attempt}/{HF_UPLOAD_ATTEMPTS}); đang thử lại...')
            time.sleep(min(2 ** attempt, 30))

    result = {
        'status': 'uploaded', 'repo_id': HF_REPO_ID, 'run_name': run_name,
        'commit_url': commit.commit_url, 'commit_oid': commit.oid,
        'files': [remote for _, remote in files],
    }
    (run_dir / 'hf_upload_result.json').write_text(
        json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('Upload thành công:', commit.commit_url)
    for _, remote in files:
        print(f'https://huggingface.co/datasets/{HF_REPO_ID}/blob/main/{quote(remote, safe="/")}')
    return result


hf_upload_result = upload_current_run_to_hf()
